# Phase 3, step 1: architecture variants (new bottlenecks) and report figures

Kaggle notebook. **It contains no report text.** It only produces numbers, CSV
files and figures. All report prose, figure captions and interpretation are
written by the team without AI assistance (`1_PROJECT_CONTEXT.md` section 1.7).
Figure axes, units and legends are set here; captions are not.

The model file `autoencoder_1D_models_torch.py` is **never modified**, so the
Phase 1 / Phase 2 reproduction path stays byte-identical. New bottlenecks arrive
through `wdn_model_ext.py` (written to disk by Part 0) plus three small edits
that this notebook applies to a **copy** of `step3`.

## Variants added

| Name | Params | What changes vs the paper |
|---|---|---|
| `attention_PE` | 111,370 | sinusoidal positional encoding before the `attention` bottleneck. **Same parameter count as `attention`**, so any EV difference is from positional information, not capacity. |
| `hybrid` | 144,906 | LSTM bottleneck on the earthquake decoder, attention+PE on the noise decoder. Selectable in the repo's `step3`, never reported. |
| `attention_LSTM` | 161,546 | 1-layer bi-LSTM as learned positional embedding, then self-attention. Also in the repo, never reported. |
| `Transformer_fixed` | 178,058 | `batch_first=True` (the original defaults to `False` and attends over the **batch** axis, not time) and `dim_feedforward=256` instead of the 2048 default. Compare against `Transformer`'s 640,394. |

## Parts

| Part | Switch | Device | What it does |
|---|---|---|---|
| 0 | always | CPU | environment, repo clone at the pinned commit, write `wdn_model_ext.py`, patch a copy of `step3` |
| 1 | `RUN_SELFTEST` | CPU | parameter counts against the five measured values, forward shapes, batch independence, positional encoding active, EV matches scikit-learn |
| 2 | `RUN_BATCHDEP` | CPU | pooled median EV of the paper's `Transformer` as a function of **evaluation batch size** and of row order. Trains nothing. |
| 3 | `RUN_SMOKE`, `RUN_TRAIN` | GPU | 2-epoch projection, then one full variant per session |
| 4 | `RUN_EVAL` | CPU | every run found, scored on the STEAD test split and the INSTANCE test split, with cluster bootstrap CIs |
| 5 | `RUN_FIGURES` | CPU | eight report figures as PDF + PNG, each backed by a CSV |

## Kaggle inputs to attach

| Dataset | Needed for |
|---|---|
| `wdn-training-data` | STEAD training and STEAD-test evaluation (attach all three HDF5 files; the two intermediate files carry the per-row SNR and earthquake names) |
| `wdn-runs-seed99` | the five STEAD-trained paper models |
| `instance-mixtures-snr40` | INSTANCE-test evaluation |
| `wdn-runs-instance-all` | the five INSTANCE-trained paper models |

Your patched `config.py`, `step3_training_1D.py` and `step3_training_1D_presplit.py`
are **embedded in Part 0** as byte copies taken from your own Phase 1 / Phase 2
notebooks, so **no `wdn_code` dataset is needed**. Part 0 prints their sha256[:16]
so you can confirm they match the recorded values (`config.py 146bcb7aa227c74e`,
`step3_training_1D.py 1e186284c55e79d2`).


Internet must be **on** for the repo clone. Set the accelerator to **GPU** only
for Part 3; Parts 1, 2, 4 and 5 run on CPU.

In [1]:
# ======================= SWITCHES: set True only for the parts you want in THIS session =======================
RUN_SELFTEST = True    # Part 1: parameter counts, shapes, batch independence, EV vs sklearn. Seconds, CPU.
RUN_BATCHDEP = True    # Part 2: paper Transformer EV vs evaluation batch size and row order. CPU, no training.
RUN_SMOKE    = False   # Part 3: 2 epochs on a subset, prints the projected full runtime
RUN_TRAIN    = False   # Part 3: full training of ONE variant (GPU session)
RUN_EVAL     = True    # Part 4: score every run found on the test splits
RUN_FIGURES  = True    # Part 5: build the figures from results/

# ----------------------------------------------------------------- training configuration
VARIANT  = "hybrid"   # one of: attention_PE, hybrid, attention_LSTM, Transformer_fixed
TRAIN_ON = "STEAD"          # "STEAD" (187,000 mixtures, recommended) or "INSTANCE" (30,000)
SEED     = 99
EPOCHS   = 100
SMOKE_EPOCHS  = 2
SMOKE_SUBSET  = 5000

# ----------------------------------------------------------------- evaluation configuration
EVAL_ON          = ("STEAD", "INSTANCE")  # test splits to score every model on
EVAL_BATCH       = 128      # fixed and recorded: Part 2 shows why this matters for the paper's Transformer
EV_CLIP          = -1.0     # component EV clipped to [EV_CLIP, 1] before averaging
N_BOOT           = 1000     # cluster bootstrap resamples (clusters = earthquake)
BOOT_SEED        = 12345
BANDPASS_HZ      = (0.5, 4.9)   # baseline band; NOT from the paper
EPOCHS_REQUIRED  = 100      # only runs that finished this many epochs are evaluated
INCLUDE_PRETRAINED = True   # also score the authors' models from the repo's trained_models/
MAX_TEST_ROWS    = None     # e.g. 8000 to subsample the STEAD test split if a session is short

# ----------------------------------------------------------------- Part 2 configuration
BATCHDEP_ROWS    = 2048                         # evenly spaced test rows used for the sweep
BATCHDEP_SIZES   = (1, 2, 8, 32, 128, 512, 2048)
BATCHDEP_PERMS   = 5                            # row orderings tried at batch size 128
BATCHDEP_MODELS  = ("Transformer", "attention")  # second one is the control

# ----------------------------------------------------------------- fixed facts
COMMIT       = "afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1"
STEAD_FILE   = "training_datasets_all_snr_40_unshuffled.hdf5"
STEAD_POHA   = "training_datasets_STEAD_plus_POHA_snr_40_unshuffled.hdf5"
STEAD_QUAKE  = "training_datasets_STEAD_waveform_snr_40.hdf5"
INSTANCE_FILE = "training_datasets_INSTANCE_snr40.hdf5"
PAPER_BOTTLENECK_NAMES = ("None", "Linear", "LSTM", "attention", "Transformer")
NEW_BOTTLENECK_NAMES   = ("attention_PE", "attention_LSTM", "hybrid", "Transformer_fixed")
# Parameter counts measured on the user's own Kaggle runs (9_PHASE1_HANDOFF.md section 6, [LOG]).
PAPER_PARAM_REF = {"None": 78090, "Linear": 86410, "LSTM": 178442,
                   "attention": 111370, "Transformer": 640394}

assert VARIANT in NEW_BOTTLENECK_NAMES, VARIANT
assert TRAIN_ON in ("STEAD", "INSTANCE"), TRAIN_ON
print("variant=%s train_on=%s seed=%s epochs=%s" % (VARIANT, TRAIN_ON, SEED, EPOCHS))
print("switches:", {k: v for k, v in sorted(globals().items())
                    if k.startswith("RUN_") and isinstance(v, bool)})

variant=hybrid train_on=STEAD seed=99 epochs=100
switches: {'RUN_BATCHDEP': True, 'RUN_EVAL': True, 'RUN_FIGURES': True, 'RUN_SELFTEST': True, 'RUN_SMOKE': False, 'RUN_TRAIN': False}


In [2]:
import os, sys, glob, json, time, re, shutil, subprocess, platform, datetime, hashlib, textwrap
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle/input")
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
TEMP = Path("/kaggle/temp") if ON_KAGGLE else (Path.cwd() / "tmp")
RESULTS, FIGURES, LOGS = WORK / "results", WORK / "figures", WORK / "logs"
for d in (RESULTS, FIGURES, LOGS, TEMP):
    d.mkdir(parents=True, exist_ok=True)

REPO = TEMP / "WaveDecompNet-paper"
SEARCH_ROOTS = [Path("/kaggle/input")] if ON_KAGGLE else []
SEARCH_ROOTS += [WORK, TEMP]


def log(msg):
    stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = "- %s  %s" % (stamp, msg)
    with open(LOGS / "lab_log.md", "a") as f:
        f.write(line + "\n")
    print(line)


def find_all(pattern, roots=None):
    """Every path under the search roots matching a glob pattern, sorted."""
    out = []
    for r in (roots if roots is not None else SEARCH_ROOTS):
        if Path(r).exists():
            out += [Path(p) for p in glob.glob(str(Path(r) / "**" / pattern), recursive=True)]
    # drop macOS resource forks and duplicates
    out = [p for p in out if not p.name.startswith("._")]
    return sorted(set(out))


def find_one(name, required=True, roots=None):
    hits = find_all(name, roots)
    if not hits:
        if required:
            raise FileNotFoundError(
                "%s not found under %s -- attach the dataset that holds it"
                % (name, [str(r) for r in (roots if roots is not None else SEARCH_ROOTS)]))
        print("NOT FOUND (optional): %s" % name)
        return None
    if len(hits) > 1:
        print("note: %d copies of %s, using %s" % (len(hits), name, hits[0]))
    return hits[0]


def sha16(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]


# ------------------------------------------------------------------ environment printout
print("python      ", platform.python_version())
try:
    import torch
    print("torch       ", torch.__version__, "| cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu         ", torch.cuda.get_device_name(0))
except Exception as exc:
    print("torch import failed:", exc)
import numpy as np
print("numpy       ", np.__version__)
print("on kaggle   ", ON_KAGGLE)
print("work dir    ", WORK)

# ------------------------------------------------------------------ repo at the pinned commit
if not (REPO / "autoencoder_1D_models_torch.py").exists():
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.run(["git", "clone", "--quiet",
                    "https://github.com/yinjiuxun/WaveDecompNet-paper.git", str(REPO)],
                   check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", COMMIT], check=True)
head = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
assert head == COMMIT, "repo is at %s, expected %s" % (head, COMMIT)
print("repo commit ", head)

# NumPy 2 removed np.Inf (Phase 1 patch)
tt = REPO / "torch_tools.py"
src = tt.read_text()
if "np.Inf" in src:
    tt.write_text(src.replace("np.Inf", "np.inf"))
    print("patched     np.Inf -> np.inf in torch_tools.py")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

log("Part 0 setup: repo %s, python %s" % (head[:12], platform.python_version()))

python       3.13.16
torch        2.11.0+cu130 | cuda available: True
gpu          Tesla T4
numpy        2.1.3
on kaggle    True
work dir     /kaggle/working
repo commit  afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1
patched     np.Inf -> np.inf in torch_tools.py
- 2026-11-17 17:09:21  Part 0 setup: repo afe2536cf29e, python 3.13.16


In [3]:
# ===================== embedded scripts: wdn_model_ext.py + your patched wdn_code =====================
# The three wdn_code files are byte copies taken from the embedded strings in your own
# Phase 1 / Phase 2 notebooks, so no wdn_code dataset has to be attached. The printed
# sha256[:16] must match the values recorded in the handoff:
#     config.py 146bcb7aa227c74e | step3_training_1D.py 1e186284c55e79d2
# `autoencoder_1D_models_torch.py` is NOT modified.
MODEL_EXT_SRC = r'''"""Extension bottlenecks for WaveDecompNet.

Repo: yinjiuxun/WaveDecompNet-paper @ afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1

This module adds new bottleneck variants WITHOUT touching
`autoencoder_1D_models_torch.py`, so the Phase 1 / Phase 2 reproduction path
stays byte-identical.

Every bottleneck here is a plain `nn.Module` taking and returning a tensor of
shape (batch, num_steps, num_features) = (B, 75, 64). `SeismogramDecoder.forward`
therefore picks them up through its generic `else: x = self.bottleneck(x)`
branch with no change to the repo file. None of these classes subclasses
`nn.LSTM` or `nn.MultiheadAttention`, which would divert the dispatch.

Variants
--------
attention_PE
    The paper's `attention` bottleneck plus a sinusoidal positional encoding.
    `nn.MultiheadAttention` with query=key=value and no positional information
    is permutation-invariant along time, so the paper's attention bottleneck
    cannot use ordering; position reaches the decoder only through the
    `x + x3` skip connection. The encoding is parameter-free, so this variant
    has EXACTLY the same parameter count as `attention` (111,370) and any
    change in EV cannot be attributed to added capacity.

attention_LSTM
    One bidirectional LSTM layer used as a learned positional embedding,
    followed by multi-head self-attention. This is the repo's own
    `Attention_bottleneck_LSTM`, which `step3` can already select by name but
    which the paper does not report. Reimplemented on `nn.MultiheadAttention`
    to avoid the softmax-axis defect in the repo's hand-rolled
    `DotProductAttention`: `F.softmax(scores, dim=0)` normalises over the
    BATCH axis instead of over keys, so attention weights do not sum to one
    over keys and, at batch size 1, every weight becomes 1.0.

hybrid
    LSTM bottleneck on the earthquake decoder, attention+PE bottleneck on the
    noise decoder. Also already selectable in `step3` and not reported.

Transformer_fixed
    The paper's `Transformer` bottleneck with two defects corrected:
      * `batch_first=True`. The original passes no `batch_first`, so it
        defaults to False and `nn.TransformerEncoder` reads the input
        (batch, steps, features) as (seq, batch, feature) -- it attends across
        the BATCH axis and never along time.
      * `dim_feedforward=256` instead of the 2048 default, a 4x expansion on a
        64-dim latent rather than 32x. The 2048 default accounts for 281,152 of
        the original 640,394 parameters.
    `norm_first=True` makes it a pre-LN block, and the same parameter-free
    positional encoding is applied, since a corrected temporal attention is
    otherwise still permutation-invariant.

The paper's five bottleneck names are deliberately NOT served by
`make_bottlenecks`; they raise `ValueError`. There is no path by which this
module can alter a reproduction run.
"""

import copy
import math

import torch
from torch import nn

__all__ = [
    "SinusoidalPositionalEncoding",
    "AttentionPEBottleneck",
    "AttentionLSTMBottleneck",
    "TransformerFixedBottleneck",
    "make_bottlenecks",
    "NEW_BOTTLENECKS",
    "PAPER_BOTTLENECKS",
]

PAPER_BOTTLENECKS = ("None", "Linear", "LSTM", "attention", "Transformer")
NEW_BOTTLENECKS = ("attention_PE", "attention_LSTM", "hybrid", "Transformer_fixed")


class SinusoidalPositionalEncoding(nn.Module):
    """Parameter-free sinusoidal positional encoding, added to the input.

    Unlike the repo's `PositionalEncoding`, the table is registered as a buffer
    (so it moves with `.to(device)` and is carried by `torch.save(model)`) and
    is built in the model dtype (the repo builds a float32 table inside a
    float64 model).
    """

    def __init__(self, d_model=64, max_len=512, dropout=0.0, dtype=torch.float64):
        super().__init__()
        if d_model % 2 != 0:
            raise ValueError("d_model must be even, got %d" % d_model)
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(1, max_len, d_model, dtype=dtype)
        position = torch.arange(max_len, dtype=dtype).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=dtype) * (-math.log(10000.0) / d_model)
        )
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, num_steps, num_features)
        if x.shape[1] > self.pe.shape[1]:
            raise ValueError(
                "sequence length %d exceeds max_len %d" % (x.shape[1], self.pe.shape[1])
            )
        return self.dropout(x + self.pe[:, : x.shape[1], :])


class AttentionPEBottleneck(nn.Module):
    """The paper's `attention` bottleneck + sinusoidal positional encoding.

    Parameter count is identical to the paper's `attention` bottleneck.
    """

    def __init__(self, d_model=64, nhead=4, dropout=0.2, pe_dropout=0.0,
                 max_len=512, dtype=torch.float64):
        super().__init__()
        self.pe = SinusoidalPositionalEncoding(d_model, max_len, pe_dropout, dtype)
        self.attention = nn.MultiheadAttention(
            d_model, nhead, dropout, batch_first=True, dtype=dtype
        )

    def forward(self, x):
        x = self.pe(x)
        x, _ = self.attention(x, x, x)
        return x


class AttentionLSTMBottleneck(nn.Module):
    """Bidirectional LSTM as positional embedding, then self-attention.

    Corrected reimplementation of the repo's `Attention_bottleneck_LSTM`.
    """

    def __init__(self, d_model=64, nhead=4, dropout=0.2, dtype=torch.float64):
        super().__init__()
        if d_model % 2 != 0:
            raise ValueError("d_model must be even, got %d" % d_model)
        self.lstm = nn.LSTM(
            d_model, d_model // 2, 1, bidirectional=True, batch_first=True, dtype=dtype
        )
        self.attention = nn.MultiheadAttention(
            d_model, nhead, dropout, batch_first=True, dtype=dtype
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        x, _ = self.attention(x, x, x)
        return x


class TransformerFixedBottleneck(nn.Module):
    """Pre-LN transformer encoder block that actually attends along time."""

    def __init__(self, d_model=64, nhead=4, dim_feedforward=256, dropout=0.1,
                 num_layers=1, use_pe=True, max_len=512, dtype=torch.float64):
        super().__init__()
        self.pe = (
            SinusoidalPositionalEncoding(d_model, max_len, 0.0, dtype) if use_pe else None
        )
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            dtype=dtype,
        )
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=num_layers, enable_nested_tensor=False
        )

    def forward(self, x):
        if self.pe is not None:
            x = self.pe(x)
        return self.encoder(x)


def _pair(module):
    """Two bottlenecks with identical initial weights.

    `step3` builds one bottleneck and `copy.deepcopy`s it for the two decoders,
    so both branches start from the same bottleneck weights. That behaviour is
    kept here so the only difference from the paper's runs is the bottleneck
    type.
    """
    return module, copy.deepcopy(module)


def make_bottlenecks(name, d_model=64, nhead=4, dropout=0.2, dtype=torch.float64):
    """Return (bottleneck_earthquake, bottleneck_noise) for an extension variant.

    Raises ValueError for the paper's five bottleneck names, which must stay on
    the original `step3` code path.
    """
    if name in PAPER_BOTTLENECKS:
        raise ValueError(
            "'%s' is one of the paper's bottlenecks and must be built by the "
            "original step3 code, not by wdn_model_ext." % name
        )

    if name == "attention_PE":
        return _pair(AttentionPEBottleneck(d_model, nhead, dropout, dtype=dtype))

    if name == "attention_LSTM":
        return _pair(AttentionLSTMBottleneck(d_model, nhead, dropout, dtype=dtype))

    if name == "Transformer_fixed":
        return _pair(TransformerFixedBottleneck(d_model, nhead, dtype=dtype))

    if name == "hybrid":
        # Repo intent (step3 'hybrid'): LSTM for earthquake, attention for noise.
        # The LSTM matches the paper's `LSTM` bottleneck (2 layers, bidirectional).
        bottleneck_earthquake = nn.LSTM(
            d_model, d_model // 2, 2, bidirectional=True, batch_first=True, dtype=dtype
        )
        bottleneck_noise = AttentionPEBottleneck(d_model, nhead, dropout, dtype=dtype)
        return bottleneck_earthquake, bottleneck_noise

    raise NameError(
        "unknown bottleneck '%s'; extension variants are %s" % (name, list(NEW_BOTTLENECKS))
    )
'''

CONFIG_PY = r'''"""Central configuration for the WaveDecompNet replication (works locally and on Kaggle).

Every other script imports this file, so paths and seeds live in exactly one place.
Override the STEAD location with the environment variable WDN_STEAD_DIR if needed.
"""
import glob
import os

ON_KAGGLE = os.path.exists("/kaggle/input")


def _find_stead(name):
    env = os.environ.get("WDN_STEAD_DIR")
    if env:
        return os.path.join(env, name)
    if ON_KAGGLE:
        hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        if hits:
            return hits[0]
    return os.path.join(os.getcwd(), "stead", name)  # local fallback


# ---------------------------------------------------------------- paths
WORK_DIR = "/kaggle/working" if ON_KAGGLE else os.getcwd()

STEAD_HDF5 = _find_stead("merged.hdf5")
STEAD_CSV = _find_stead("merged.csv")

DATASET_DIR = os.path.join(WORK_DIR, "training_datasets")
POHA_DIR = os.path.join(WORK_DIR, "continuous_waveforms")
POHA_MSEED = os.path.join(POHA_DIR, "IU.POHA.00.20210601-20210801.mseed")

STEAD_MIX_H5 = os.path.join(DATASET_DIR, "training_datasets_STEAD_waveform_snr_40.hdf5")
POHA_MIX_H5 = os.path.join(DATASET_DIR, "training_datasets_STEAD_plus_POHA_snr_40_unshuffled.hdf5")
# paper-faithful name (the file is actually shuffled WITH replacement, see step2d)
FINAL_DATASET = os.path.join(DATASET_DIR, "training_datasets_all_snr_40_unshuffled.hdf5")
FINAL_DATASET_PERMUTED = os.path.join(DATASET_DIR, "training_datasets_all_snr_40_permuted.hdf5")

MODEL_DIR = os.path.join(WORK_DIR, "Model_and_datasets_1D_all_snr_40_unshuffled_equal_epoch100")

# ---------------------------------------------------------------- data recipe (from the repo)
SNR_MIN_DB = 40
N_EVENTS = 100000                 # step2b: number of STEAD mixtures
F_DOWNSAMPLE = 10                 # Hz
N_APPEND = 1000                   # chunk size when writing HDF5
STEAD_SEEDS = dict(earthquake=101, noise=102, shift=103, snr=104)   # step2b
POHA_SEEDS = dict(earthquake=111, noise=112, shift=113, snr=114)    # step2c
POHA_AMPLITUDE_THRESHOLD = 6      # step2c: drop samples above 6 x median amplitude
POHA_EXPECTED_FS = 40.0           # step2c calls decimate(4) -> 10 Hz, so it assumes 40 Hz

# step2c in the original code writes only full chunks of N_APPEND, so the last
# (N mod 1000) windows are silently dropped. False = keep that behaviour (fidelity).
KEEP_REMAINDER = False

# step2d in the original code shuffles with np.random.choice(N, N): WITH replacement,
# no seed. True = same behaviour but seeded and logged; False = a proper permutation.
SHUFFLE_WITH_REPLACEMENT = True
SHUFFLE_SEED = 0

# ---------------------------------------------------------------- training (step3)
BATCH_SIZE = 128
EPOCHS = 100
LR = 1e-3
MIN_EPOCHS = 30
PATIENCE = None                   # None = no early stopping (always 100 epochs)
TORCH_SEED = 99
SPLIT_SEED_1 = 13
SPLIT_SEED_2 = 20
TRAIN_SIZE = 0.6
TEST_SIZE = 0.5                   # of the remaining 40 % -> 20 % val, 20 % test
'''

STEP3_PY = r'''"""step3 (patched): train ONE bottleneck per run (needed for Kaggle's session limits).

Same model, loss, optimiser, LR schedule, split (60/20/20, seeds 13 and 20) and seeds as the repo.
Changes: command-line arguments, paths from config.py, no `tabnanny` import, no `verbose=` in
LambdaLR (removed in newer PyTorch), figure saved headless, run info saved as JSON.

Quick smoke test first (a few minutes):
    python step3_training_1D.py --bottleneck None --epochs 2 --subset 5000 --tag _smoke
Full run:
    python step3_training_1D.py --bottleneck LSTM
"""
import argparse
import copy
import json
import os
import random
import time

import h5py
import matplotlib
matplotlib.use("Agg")
import numpy as np
import torch
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

import config as C
from autoencoder_1D_models_torch import SeismogramDecoder, SeismogramEncoder, SeisSeparator
from torch_tools import WaveformDataset, training_loop_branches, try_gpu

BOTTLENECKS = ["None", "Linear", "LSTM", "attention", "Transformer"]


def make_bottleneck(name):
    if name == "None":
        return None
    if name == "Linear":
        return torch.nn.Linear(64, 64, dtype=torch.float64)
    if name == "LSTM":
        return torch.nn.LSTM(64, 32, 2, bidirectional=True, batch_first=True, dtype=torch.float64)
    if name == "attention":
        return torch.nn.MultiheadAttention(64, 4, 0.2, batch_first=True, dtype=torch.float64)
    if name == "Transformer":
        layer = torch.nn.TransformerEncoderLayer(d_model=64, nhead=4, dtype=torch.float64)
        return torch.nn.TransformerEncoder(layer, num_layers=1)
    raise NameError("bottleneck type has not been defined!")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--bottleneck", required=True, choices=BOTTLENECKS)
    p.add_argument("--data", default=C.FINAL_DATASET)
    p.add_argument("--seed", type=int, default=C.TORCH_SEED)
    p.add_argument("--epochs", type=int, default=C.EPOCHS)
    p.add_argument("--subset", type=int, default=None, help="use only the first N samples (smoke test)")
    p.add_argument("--tag", default=None, help="suffix of the output folder")
    args = p.parse_args()

    tag = args.tag if args.tag is not None else ("_warmup" if args.seed == C.TORCH_SEED else f"_warmup_seed{args.seed}")
    model_structure = "Branch_Encoder_Decoder"
    model_name = f"{model_structure}_{args.bottleneck}"
    out_dir = os.path.join(C.MODEL_DIR, model_name + tag)
    os.makedirs(out_dir, exist_ok=True)
    device = try_gpu()
    print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

    # ---- data
    print("#" * 12 + " Loading data " + "#" * 12)
    with h5py.File(args.data, "r") as f:
        n = f["X_train"].shape[0] if args.subset is None else min(args.subset, f["X_train"].shape[0])
        X_train, Y_train = f["X_train"][:n], f["Y_train"][:n]
    print("data:", X_train.shape, X_train.dtype)

    X_training, X_test, Y_training, Y_test = train_test_split(
        X_train, Y_train, train_size=C.TRAIN_SIZE, random_state=C.SPLIT_SEED_1)
    X_validate, X_test, Y_validate, Y_test = train_test_split(
        X_test, Y_test, test_size=C.TEST_SIZE, random_state=C.SPLIT_SEED_2)
    del X_train, Y_train, X_test, Y_test

    # ---- seeds (same as the repo, torch seed configurable for multi-seed runs)
    torch.manual_seed(args.seed)
    random.seed(0)
    np.random.seed(20)
    torch.backends.cudnn.benchmark = False

    training_data = WaveformDataset(X_training, Y_training)
    validate_data = WaveformDataset(X_validate, Y_validate)

    # ---- model
    bottleneck = make_bottleneck(args.bottleneck)
    encoder = SeismogramEncoder()
    decoder_earthquake = SeismogramDecoder(bottleneck=copy.deepcopy(bottleneck))
    decoder_noise = SeismogramDecoder(bottleneck=copy.deepcopy(bottleneck))
    model = SeisSeparator(model_name, encoder, decoder_earthquake, decoder_noise).to(device=device)
    n_params = int(sum(p.numel() for p in model.parameters()))
    print(f"{model_name}: {n_params} parameters")

    batch_size, lr = C.BATCH_SIZE, C.LR
    loss_fn = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    LR_func = lambda epoch: (epoch + 1) / 11 if (epoch <= 10) else (0.95 ** epoch if (epoch < 50) else 0.95 ** 50)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=LR_func)

    train_iter = DataLoader(training_data, batch_size=batch_size, shuffle=True)
    validate_iter = DataLoader(validate_data, batch_size=batch_size, shuffle=False)

    # ---- train
    print("#" * 12 + " training model " + model_name + " " + "#" * 12)
    t0 = time.time()
    model, avg_train_losses, avg_valid_losses, partial_loss = training_loop_branches(
        train_iter, validate_iter, model, loss_fn, optimizer, scheduler,
        epochs=args.epochs, patience=C.PATIENCE, device=device, minimum_epochs=C.MIN_EPOCHS)
    runtime = time.time() - t0
    print(f"Training is done! {runtime / 60:.1f} min")

    # ---- save
    torch.save(model, os.path.join(out_dir, f"{model_name}_Model.pth"))
    with h5py.File(os.path.join(out_dir, f"{model_name}_Training_history.hdf5"), "w") as f:
        f.create_dataset("loss", data=avg_train_losses)
        f.create_dataset("val_loss", data=avg_valid_losses)
        f.create_dataset("earthquake_loss", data=partial_loss[0])
        f.create_dataset("earthquake_val_loss", data=partial_loss[1])
        f.create_dataset("noise_loss", data=partial_loss[2])
        f.create_dataset("noise_val_loss", data=partial_loss[3])
    with h5py.File(os.path.join(out_dir, f"{model_name}_Dataset_split.hdf5"), "w") as f:
        f.attrs["model_name"] = model_name
        f.attrs["train_size"] = C.TRAIN_SIZE
        f.attrs["test_size"] = C.TEST_SIZE
        f.attrs["rand_seed1"] = C.SPLIT_SEED_1
        f.attrs["rand_seed2"] = C.SPLIT_SEED_2
    with open(os.path.join(out_dir, "run_info.json"), "w") as f:
        json.dump(dict(model_name=model_name, bottleneck=args.bottleneck, torch_seed=args.seed,
                       epochs=args.epochs, subset=args.subset, data=args.data,
                       n_train=len(training_data), n_val=len(validate_data), n_params=n_params,
                       runtime_s=runtime, device=str(device),
                       gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else None,
                       torch=torch.__version__, final_train_loss=float(avg_train_losses[-1]),
                       final_val_loss=float(avg_valid_losses[-1])), f, indent=2)

    plt.figure()
    plt.plot(avg_train_losses, "o", label="loss")
    plt.plot(avg_valid_losses, "-", label="Validation loss")
    for k, (lab, mk) in enumerate(zip(["earthquake train loss", "earthquake valid loss",
                                       "noise train loss", "noise valid loss"], ["o", "", "o", ""])):
        plt.plot(partial_loss[k], marker=mk, label=lab)
    plt.yscale("log")
    plt.legend()
    plt.title(model_name)
    plt.savefig(os.path.join(out_dir, f"{model_name}_Training_history.png"))
    plt.close()
    print("saved to", out_dir)


if __name__ == "__main__":
    main()
'''

STEP3_PRESPLIT_PY = r'''"""step3_presplit = step3 (patched) + ONE new option: --presplit.

--presplit uses the dataset split stored in the data file (dataset `split_id`: 0 = train, 1 = validation, 2 = test) instead of the
random 60/20/20 split after loading. Needed for the INSTANCE data, which is split by earthquake BEFORE mixing. Without --presplit the
behaviour is identical to step3_training_1D.py.

Original docstring:
step3 (patched): train ONE bottleneck per run (needed for Kaggle's session limits).

Same model, loss, optimiser, LR schedule, split (60/20/20, seeds 13 and 20) and seeds as the repo.
Changes: command-line arguments, paths from config.py, no `tabnanny` import, no `verbose=` in
LambdaLR (removed in newer PyTorch), figure saved headless, run info saved as JSON.

Quick smoke test first (a few minutes):
    python step3_training_1D.py --bottleneck None --epochs 2 --subset 5000 --tag _smoke
Full run:
    python step3_training_1D.py --bottleneck LSTM
"""
import argparse
import copy
import json
import os
import random
import time

import h5py
import matplotlib
matplotlib.use("Agg")
import numpy as np
import torch
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

import config as C
from autoencoder_1D_models_torch import SeismogramDecoder, SeismogramEncoder, SeisSeparator
from torch_tools import WaveformDataset, training_loop_branches, try_gpu

BOTTLENECKS = ["None", "Linear", "LSTM", "attention", "Transformer"]


def make_bottleneck(name):
    if name == "None":
        return None
    if name == "Linear":
        return torch.nn.Linear(64, 64, dtype=torch.float64)
    if name == "LSTM":
        return torch.nn.LSTM(64, 32, 2, bidirectional=True, batch_first=True, dtype=torch.float64)
    if name == "attention":
        return torch.nn.MultiheadAttention(64, 4, 0.2, batch_first=True, dtype=torch.float64)
    if name == "Transformer":
        layer = torch.nn.TransformerEncoderLayer(d_model=64, nhead=4, dtype=torch.float64)
        return torch.nn.TransformerEncoder(layer, num_layers=1)
    raise NameError("bottleneck type has not been defined!")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--bottleneck", required=True, choices=BOTTLENECKS)
    p.add_argument("--data", default=C.FINAL_DATASET)
    p.add_argument("--seed", type=int, default=C.TORCH_SEED)
    p.add_argument("--epochs", type=int, default=C.EPOCHS)
    p.add_argument("--subset", type=int, default=None, help="use only the first N samples (smoke test)")
    p.add_argument("--tag", default=None, help="suffix of the output folder")
    p.add_argument("--presplit", action="store_true", help="use split_id stored in the data file (0 train, 1 val, 2 test)")
    args = p.parse_args()

    tag = args.tag if args.tag is not None else ("_warmup" if args.seed == C.TORCH_SEED else f"_warmup_seed{args.seed}")
    model_structure = "Branch_Encoder_Decoder"
    model_name = f"{model_structure}_{args.bottleneck}"
    out_dir = os.path.join(C.MODEL_DIR, model_name + tag)
    os.makedirs(out_dir, exist_ok=True)
    device = try_gpu()
    print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

    # ---- data
    print("#" * 12 + " Loading data " + "#" * 12)
    if args.presplit:
        with h5py.File(args.data, "r") as f:
            n_all = f["X_train"].shape[0]
            assert "split_id" in f, "--presplit needs a split_id dataset in the data file"
            sid_all = f["split_id"][:]
            if args.subset is None or args.subset >= n_all:
                sel = np.arange(n_all)
                X_train, Y_train = f["X_train"][:], f["Y_train"][:]
            else:                                    # evenly spaced rows keep the train / val / test proportions
                sel = np.unique(np.linspace(0, n_all - 1, args.subset).astype(int))
                X_train, Y_train = f["X_train"][sel], f["Y_train"][sel]
            sid = sid_all[sel]
        print("data:", X_train.shape, X_train.dtype, "| presplit rows train/val/test:", [int((sid == k).sum()) for k in range(3)])
        assert (sid == 0).any() and (sid == 1).any(), "no train or validation rows selected"
        X_training, Y_training = X_train[sid == 0], Y_train[sid == 0]
        X_validate, Y_validate = X_train[sid == 1], Y_train[sid == 1]
        del X_train, Y_train
    else:
        with h5py.File(args.data, "r") as f:
            n = f["X_train"].shape[0] if args.subset is None else min(args.subset, f["X_train"].shape[0])
            X_train, Y_train = f["X_train"][:n], f["Y_train"][:n]
        print("data:", X_train.shape, X_train.dtype)

        X_training, X_test, Y_training, Y_test = train_test_split(
            X_train, Y_train, train_size=C.TRAIN_SIZE, random_state=C.SPLIT_SEED_1)
        X_validate, X_test, Y_validate, Y_test = train_test_split(
            X_test, Y_test, test_size=C.TEST_SIZE, random_state=C.SPLIT_SEED_2)
        del X_train, Y_train, X_test, Y_test

    # ---- seeds (same as the repo, torch seed configurable for multi-seed runs)
    torch.manual_seed(args.seed)
    random.seed(0)
    np.random.seed(20)
    torch.backends.cudnn.benchmark = False

    training_data = WaveformDataset(X_training, Y_training)
    validate_data = WaveformDataset(X_validate, Y_validate)

    # ---- model
    bottleneck = make_bottleneck(args.bottleneck)
    encoder = SeismogramEncoder()
    decoder_earthquake = SeismogramDecoder(bottleneck=copy.deepcopy(bottleneck))
    decoder_noise = SeismogramDecoder(bottleneck=copy.deepcopy(bottleneck))
    model = SeisSeparator(model_name, encoder, decoder_earthquake, decoder_noise).to(device=device)
    n_params = int(sum(p.numel() for p in model.parameters()))
    print(f"{model_name}: {n_params} parameters")

    batch_size, lr = C.BATCH_SIZE, C.LR
    loss_fn = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    LR_func = lambda epoch: (epoch + 1) / 11 if (epoch <= 10) else (0.95 ** epoch if (epoch < 50) else 0.95 ** 50)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=LR_func)

    train_iter = DataLoader(training_data, batch_size=batch_size, shuffle=True)
    validate_iter = DataLoader(validate_data, batch_size=batch_size, shuffle=False)

    # ---- train
    print("#" * 12 + " training model " + model_name + " " + "#" * 12)
    t0 = time.time()
    model, avg_train_losses, avg_valid_losses, partial_loss = training_loop_branches(
        train_iter, validate_iter, model, loss_fn, optimizer, scheduler,
        epochs=args.epochs, patience=C.PATIENCE, device=device, minimum_epochs=C.MIN_EPOCHS)
    runtime = time.time() - t0
    print(f"Training is done! {runtime / 60:.1f} min")

    # ---- save
    torch.save(model, os.path.join(out_dir, f"{model_name}_Model.pth"))
    with h5py.File(os.path.join(out_dir, f"{model_name}_Training_history.hdf5"), "w") as f:
        f.create_dataset("loss", data=avg_train_losses)
        f.create_dataset("val_loss", data=avg_valid_losses)
        f.create_dataset("earthquake_loss", data=partial_loss[0])
        f.create_dataset("earthquake_val_loss", data=partial_loss[1])
        f.create_dataset("noise_loss", data=partial_loss[2])
        f.create_dataset("noise_val_loss", data=partial_loss[3])
    with h5py.File(os.path.join(out_dir, f"{model_name}_Dataset_split.hdf5"), "w") as f:
        f.attrs["model_name"] = model_name
        f.attrs["train_size"] = C.TRAIN_SIZE
        f.attrs["test_size"] = C.TEST_SIZE
        f.attrs["rand_seed1"] = C.SPLIT_SEED_1
        f.attrs["rand_seed2"] = C.SPLIT_SEED_2
        f.attrs["presplit"] = bool(args.presplit)
    with open(os.path.join(out_dir, "run_info.json"), "w") as f:
        json.dump(dict(model_name=model_name, bottleneck=args.bottleneck, torch_seed=args.seed,
                       epochs=args.epochs, subset=args.subset, data=args.data, presplit=bool(args.presplit),
                       n_train=len(training_data), n_val=len(validate_data), n_params=n_params,
                       runtime_s=runtime, device=str(device),
                       gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else None,
                       torch=torch.__version__, final_train_loss=float(avg_train_losses[-1]),
                       final_val_loss=float(avg_valid_losses[-1])), f, indent=2)

    plt.figure()
    plt.plot(avg_train_losses, "o", label="loss")
    plt.plot(avg_valid_losses, "-", label="Validation loss")
    for k, (lab, mk) in enumerate(zip(["earthquake train loss", "earthquake valid loss",
                                       "noise train loss", "noise valid loss"], ["o", "", "o", ""])):
        plt.plot(partial_loss[k], marker=mk, label=lab)
    plt.yscale("log")
    plt.legend()
    plt.title(model_name)
    plt.savefig(os.path.join(out_dir, f"{model_name}_Training_history.png"))
    plt.close()
    print("saved to", out_dir)


if __name__ == "__main__":
    main()
'''

import importlib
import difflib

# wdn_model_ext goes to both locations: torch.save(model) pickles the class path, so
# loading any extension model later needs it importable, including in an eval-only session.
for _dest in (REPO, WORK):
    (_dest / "wdn_model_ext.py").write_text(MODEL_EXT_SRC)
ext_path = REPO / "wdn_model_ext.py"
print("wdn_model_ext.py -> %s and %s   sha256[:16]=%s" % (REPO, WORK, sha16(ext_path)))

importlib.invalidate_caches()
import wdn_model_ext as _wme
importlib.reload(_wme)
print("  importable from %s; variants: %s" % (_wme.__file__, list(_wme.NEW_BOTTLENECKS)))

(REPO / "config.py").write_text(CONFIG_PY)
print("config.py               sha256[:16]=%s  (handoff: 146bcb7aa227c74e)"
      % sha16(REPO / "config.py"))

_step3_src = STEP3_PRESPLIT_PY if TRAIN_ON == "INSTANCE" else STEP3_PY
_step3_name = "step3_training_1D_presplit.py" if TRAIN_ON == "INSTANCE" else "step3_training_1D.py"
(REPO / _step3_name).write_text(_step3_src)
print("%-24s sha256[:16]=%s  (unpatched original)"
      % (_step3_name, sha16(REPO / _step3_name)))


# ===================== the three edits =====================
def patch_step3(text):
    """Teach step3 the extension bottlenecks. Three edits, matched on text, not line
    numbers: (1) import, (2) widen the argparse `choices` list, (3) build the two
    decoders from a bottleneck pair when the name is an extension variant."""
    notes = []
    if "wdn_model_ext" in text:
        return text, ["already patched; left unchanged"]
    lines = text.split("\n")

    # --- edit 1: import, right after the autoencoder import statement
    hit = [i for i, l in enumerate(lines)
           if "SeisSeparator" in l and l.lstrip().startswith(("from ", "import "))]
    if not hit:
        raise RuntimeError("edit 1 anchor not found: no import line mentions SeisSeparator")
    lines.insert(hit[0] + 1, "from wdn_model_ext import make_bottlenecks, NEW_BOTTLENECKS")
    notes.append("edit 1: import added after line %d" % (hit[0] + 1))

    # --- edit 2: argparse uses choices=BOTTLENECKS, so the new names must be in that list
    hit = [i for i, l in enumerate(lines) if re.match(r"\s*BOTTLENECKS\s*=\s*\[", l)]
    if not hit:
        raise RuntimeError("edit 2 anchor not found: no `BOTTLENECKS = [...]` assignment")
    i = hit[0]
    lines[i] = lines[i].rstrip() + " + list(NEW_BOTTLENECKS)"
    notes.append("edit 2: BOTTLENECKS widened at line %d" % (i + 1))

    # --- edit 3: the model-assembly block
    text2 = "\n".join(lines)
    blk = re.search(
        r"(?P<ind>[ \t]*)bottleneck = make_bottleneck\(args\.bottleneck\)\n"
        r"(?P=ind)encoder = SeismogramEncoder\(\)\n"
        r"(?P=ind)decoder_earthquake = SeismogramDecoder\(bottleneck=copy\.deepcopy\(bottleneck\)\)\n"
        r"(?P=ind)decoder_noise = SeismogramDecoder\(bottleneck=copy\.deepcopy\(bottleneck\)\)\n",
        text2)
    if blk is None:
        raise RuntimeError("edit 3 anchor not found: the make_bottleneck / SeismogramDecoder block")
    ind = blk.group("ind")
    repl = (
        "%sif args.bottleneck in NEW_BOTTLENECKS:\n"
        "%s    _bn_eq, _bn_noise = make_bottlenecks(args.bottleneck)\n"
        "%selse:\n"
        "%s    bottleneck = make_bottleneck(args.bottleneck)\n"
        "%s    _bn_eq, _bn_noise = copy.deepcopy(bottleneck), copy.deepcopy(bottleneck)\n"
        "%sencoder = SeismogramEncoder()\n"
        "%sdecoder_earthquake = SeismogramDecoder(bottleneck=_bn_eq)\n"
        "%sdecoder_noise = SeismogramDecoder(bottleneck=_bn_noise)\n" % ((ind,) * 8))
    text2 = text2[:blk.start()] + repl + text2[blk.end():]
    notes.append("edit 3: model assembly rewritten at line %d"
                 % (text2[:blk.start()].count("\n") + 1))
    return text2, notes


_new, _notes = patch_step3(_step3_src)
STEP3 = REPO / "step3_training_1D_ext.py"
STEP3.write_text(_new)
for _n in _notes:
    print(" ", _n)
print("\n%s  sha256[:16]=%s" % (STEP3.name, sha16(STEP3)))

print("\n--- diff (patched copy only; your original is untouched) ---")
for _d in difflib.unified_diff(_step3_src.split("\n"), _new.split("\n"),
                               fromfile=_step3_name, tofile=STEP3.name, lineterm="", n=2):
    print(_d)

compile(_new, str(STEP3), "exec")
for _tok in ("wdn_model_ext", "NEW_BOTTLENECKS", "_bn_eq"):
    assert _tok in _new, "patch verification failed: %s missing" % _tok
STEP3_FLAGS = sorted(set(re.findall(r"add_argument\(\s*['\"]--([A-Za-z0-9_\-]+)", _new)))
print("\npatch verified; compiles. step3 flags: %s" % STEP3_FLAGS)
assert "bottleneck" in STEP3_FLAGS
log("Part 0: wdn_model_ext (%s) + %s (%s)" % (sha16(ext_path), STEP3.name, sha16(STEP3)))

wdn_model_ext.py -> /kaggle/temp/WaveDecompNet-paper and /kaggle/working   sha256[:16]=65dea6e1eb352830
  importable from /kaggle/temp/WaveDecompNet-paper/wdn_model_ext.py; variants: ['attention_PE', 'attention_LSTM', 'hybrid', 'Transformer_fixed']
config.py               sha256[:16]=146bcb7aa227c74e  (handoff: 146bcb7aa227c74e)
step3_training_1D.py     sha256[:16]=1e186284c55e79d2  (unpatched original)
  edit 1: import added after line 29
  edit 2: BOTTLENECKS widened at line 33
  edit 3: model assembly rewritten at line 92

step3_training_1D_ext.py  sha256[:16]=d0466df5a716ccb9

--- diff (patched copy only; your original is untouched) ---
--- step3_training_1D.py
+++ step3_training_1D_ext.py
@@ -28,7 +28,8 @@
 import config as C
 from autoencoder_1D_models_torch import SeismogramDecoder, SeismogramEncoder, SeisSeparator
+from wdn_model_ext import make_bottlenecks, NEW_BOTTLENECKS
 from torch_tools import WaveformDataset, training_loop_branches, try_gpu
 
-BOTTLENECKS = ["None", "Line

## Helpers (always run, cheap)

Metric definitions follow the repo's `step5` and the Phase 1 / Phase 2
evaluation, so new numbers are directly comparable with
`table_E1_E2_E3.csv`:

- **EV** = `sklearn.explained_variance_score(multioutput='raw_values')`, one
  value per trace and component, implemented vectorised and checked against
  scikit-learn in Part 1.
- Noise truth = input minus true earthquake; noise prediction = second decoder
  output.
- **Pooled median EV** = median over all rows x 3 components.
- **Mean clipped EV** = mean of component EV clipped to `[EV_CLIP, 1]`. The raw
  mean is meaningless: near-zero-variance targets produce values around -1e58.
- **Test loss** = `mean(MSE earthquake + MSE noise)`.
- **Consistency residual** = `mean((pred_eq + pred_noise - x)^2) / mean(x^2)`.
  Nothing in the architecture or the loss ties the two decoders together, even
  though the data satisfies `earthquake + noise = mixture` exactly by
  construction.
- CIs come from a **cluster bootstrap over earthquakes**, because rows from one
  earthquake are correlated. The same resample draws are reused for every model,
  which is what makes the paired comparisons valid. They cover test-sample
  variation only, **not** training-seed variation: one seed is used throughout.

Every model is scored on exactly the same rows, in float64, in fixed batches of
`EVAL_BATCH`, read in the same order. Part 2 shows why the batch size has to be
pinned and recorded.

In [4]:
import numpy as np
import pandas as pd
import h5py

READ_BLOCK = 20000          # rows per HDF5 read
BOOT_Q = (2.5, 97.5)


def read_csv_keep(path):
    """pandas reads the string 'None' (a bottleneck name) as missing by default."""
    return pd.read_csv(path, keep_default_na=False, na_values=[""])


# ---------------------------------------------------------------- metrics
def ev_raw(y, p):
    """Explained variance per row and component. y, p: (n, T, 3) -> (n, 3).

    Mirrors sklearn.explained_variance_score(multioutput='raw_values'),
    including its handling of a zero denominator.
    """
    d = y - p
    num = d.var(axis=1)
    den = y.var(axis=1)
    out = np.ones_like(den)
    nz_den, nz_num = den != 0, num != 0
    v = nz_den & nz_num
    out[v] = 1.0 - num[v] / den[v]
    out[nz_num & ~nz_den] = 0.0
    return out


def clipped_mean(ev, lo=None):
    """Row-wise mean of component EV after clipping to [lo, 1]."""
    lo = EV_CLIP if lo is None else lo
    return np.clip(ev, lo, 1.0).mean(axis=1)


def row_corr(y, p):
    """Pearson correlation per row over the flattened (T x 3) vector."""
    a = y.reshape(len(y), -1)
    b = p.reshape(len(p), -1)
    a = a - a.mean(axis=1, keepdims=True)
    b = b - b.mean(axis=1, keepdims=True)
    na = np.sqrt((a * a).sum(axis=1))
    nb = np.sqrt((b * b).sum(axis=1))
    den = na * nb
    out = np.zeros(len(a))
    ok = den > 0
    out[ok] = (a[ok] * b[ok]).sum(axis=1) / den[ok]
    return out


# ---------------------------------------------------------------- cluster bootstrap
def make_boot_indices(clusters, n_boot=None, seed=None):
    """Row indices for `n_boot` bootstrap resamples of whole earthquakes."""
    n_boot = N_BOOT if n_boot is None else n_boot
    seed = BOOT_SEED if seed is None else seed
    clusters = np.asarray(clusters)
    uniq, inv = np.unique(clusters, return_inverse=True)
    order = np.argsort(inv, kind="stable")
    sorted_inv = inv[order]
    lo = np.searchsorted(sorted_inv, np.arange(len(uniq)), side="left")
    hi = np.searchsorted(sorted_inv, np.arange(len(uniq)), side="right")
    groups = [order[lo[k]:hi[k]] for k in range(len(uniq))]
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(groups), len(groups))
        out.append(np.concatenate([groups[k] for k in pick]).astype(np.int32))
    print("bootstrap: %d resamples over %d clusters (%d rows)"
          % (n_boot, len(uniq), len(clusters)))
    return out


def boot_ci(stat_fn, boot_idx):
    """(lo, hi) percentile CI of stat_fn evaluated on each resample."""
    vals = np.fromiter((stat_fn(ix) for ix in boot_idx), dtype=np.float64,
                       count=len(boot_idx))
    return float(np.percentile(vals, BOOT_Q[0])), float(np.percentile(vals, BOOT_Q[1]))


# ---------------------------------------------------------------- predictors
def bandpass_filter(x, lo_hi=None, fs=10.0, order=4):
    from scipy.signal import butter, sosfiltfilt
    lo, hi = BANDPASS_HZ if lo_hi is None else lo_hi
    sos = butter(order, [lo, hi], btype="bandpass", fs=fs, output="sos")
    return sosfiltfilt(sos, x, axis=1)


def make_predictor(kind, model=None):
    """Uniform interface: f(x_batch (b,T,3)) -> (eq, noise), both (b,T,3)."""
    if kind == "identity":
        return lambda xb: (xb.copy(), xb.copy())
    if kind == "bandpass":
        def f(xb):
            eq = np.ascontiguousarray(bandpass_filter(xb))
            return eq, xb - eq
        return f

    import torch

    def f(xb):
        with torch.no_grad():
            t = torch.from_numpy(np.ascontiguousarray(np.moveaxis(xb, -1, 1)))
            a, b = model(t.to(torch.float64))
            return (np.moveaxis(a.detach().numpy(), 1, -1),
                    np.moveaxis(b.detach().numpy(), 1, -1))
    return f


def load_model(path):
    import torch
    m = torch.load(str(path), map_location="cpu", weights_only=False)
    m.eval()
    m.to(torch.float64)
    return m


def n_params(m):
    return int(sum(p.numel() for p in m.parameters()))


# ---------------------------------------------------------------- scoring
def eval_predictor(pred_fn, X, Y, batch=None, keep_rows=()):
    """Per-row metrics for one predictor on fixed, ordered batches.

    Returns a dict of arrays plus `waveforms` for the row positions in
    `keep_rows` (used by the example-waveform figure).
    """
    batch = EVAL_BATCH if batch is None else batch
    n = len(X)
    out = {k: np.empty(n) for k in ("mse_eq", "mse_noise", "corr_eq", "resid")}
    out["ev_eq"] = np.empty((n, X.shape[2]))
    out["ev_noise"] = np.empty((n, X.shape[2]))
    keep = set(int(k) for k in keep_rows)
    waveforms = {}
    for s in range(0, n, batch):
        e = min(s + batch, n)
        xb, yb = X[s:e], Y[s:e]
        nb = xb - yb                                  # noise truth
        peq, pnz = pred_fn(xb)
        out["ev_eq"][s:e] = ev_raw(yb, peq)
        out["ev_noise"][s:e] = ev_raw(nb, pnz)
        out["mse_eq"][s:e] = ((yb - peq) ** 2).mean(axis=(1, 2))
        out["mse_noise"][s:e] = ((nb - pnz) ** 2).mean(axis=(1, 2))
        out["corr_eq"][s:e] = row_corr(yb, peq)
        denom = (xb ** 2).mean(axis=(1, 2))
        resid = ((peq + pnz - xb) ** 2).mean(axis=(1, 2))
        out["resid"][s:e] = np.where(denom > 0, resid / np.maximum(denom, 1e-300), 0.0)
        for k in keep:
            if s <= k < e:
                waveforms[k] = dict(x=xb[k - s].copy(), y=yb[k - s].copy(),
                                    peq=peq[k - s].copy(), pnz=pnz[k - s].copy())
    out["ev_eq_clipped"] = clipped_mean(out["ev_eq"])
    out["ev_noise_clipped"] = clipped_mean(out["ev_noise"])
    out["waveforms"] = waveforms
    return out


# ---------------------------------------------------------------- test splits
def stead_split_indices(n):
    """Rebuild step3's split on row indices: train_size=0.6/seed 13, then 50/50/seed 20."""
    from sklearn.model_selection import train_test_split
    idx = np.arange(n)
    tr, rest = train_test_split(idx, train_size=0.6, random_state=13)
    va, te = train_test_split(rest, test_size=0.5, random_state=20)
    return tr, va, te


def load_rows(h5path, row_idx, x_key="X_train", y_key="Y_train"):
    """Read the given rows of X and Y in file order, block by block."""
    ri = np.sort(np.asarray(row_idx))
    with h5py.File(h5path, "r") as f:
        X, Y = f[x_key], f[y_key]
        n, T, C = X.shape
        Xo = np.empty((len(ri), T, C), dtype=np.float64)
        Yo = np.empty_like(Xo)
        w = 0
        for s in range(0, n, READ_BLOCK):
            e = min(s + READ_BLOCK, n)
            m = ri[(ri >= s) & (ri < e)]
            if len(m) == 0:
                continue
            xb, yb = X[s:e], Y[s:e]
            Xo[w:w + len(m)] = xb[m - s]
            Yo[w:w + len(m)] = yb[m - s]
            w += len(m)
    assert w == len(ri)
    return ri, Xo, Yo


# ---------------------------------------------------------------- run discovery
def parse_run_folder(name):
    """'Branch_Encoder_Decoder_<bottleneck><suffixes>' -> (bottleneck, trained_on, seed, kind)."""
    prefix = "Branch_Encoder_Decoder_"
    if not name.startswith(prefix):
        return None
    rest = name[len(prefix):]
    kind = "smoke" if "_smoke" in rest else "full"
    trained_on = "INSTANCE" if "_instance" in rest else "STEAD"
    b, seed = rest, 99
    m = re.search(r"_seed(\d+)$", b)
    if m:
        seed = int(m.group(1))
        b = b[:m.start()]
    for suf in ("_instance", "_warmup", "_smoke"):
        if b.endswith(suf):
            b = b[:-len(suf)]
    b = b.replace("_instance", "").replace("_warmup", "").replace("_smoke", "")
    return b, trained_on, seed, kind


def model_id(bottleneck, trained_on, seed):
    return "%s_%s_seed%d" % (trained_on.lower(), bottleneck, seed)


def discover_runs(require_epochs=None):
    """Every finished training run under the search roots, keyed by model id."""
    require_epochs = EPOCHS_REQUIRED if require_epochs is None else require_epochs
    runs = {}
    for pth in find_all("*_Model.pth"):
        # The cloned repo ships the authors' own weights in trained_models/, whose
        # folder names look exactly like ours but carry no _warmup / _instance
        # suffix. Without this guard they would be picked up as our STEAD runs.
        if REPO in pth.parents:
            continue
        parsed = parse_run_folder(pth.parent.name)
        if parsed is None:
            continue
        b, trained_on, seed, kind = parsed
        if kind == "smoke":
            continue
        hist = sorted(pth.parent.glob("*_Training_history.hdf5"))
        epochs = None
        if hist:
            try:
                with h5py.File(hist[0], "r") as f:
                    epochs = int(len(f["loss"][:]))
            except Exception:
                epochs = None
        if require_epochs and (epochs is None or epochs < require_epochs):
            print("skip %-44s (epochs=%s)" % (pth.parent.name, epochs))
            continue
        mid = model_id(b, trained_on, seed)
        if mid in runs:
            continue
        info = {}
        for jf in ("run_info.json",):
            jp = pth.parent / jf
            if jp.exists():
                try:
                    info.update(json.loads(jp.read_text()))
                except Exception:
                    pass
        runs[mid] = dict(model_id=mid, bottleneck=b, trained_on=trained_on, seed=seed,
                         path=pth, folder=pth.parent, epochs=epochs,
                         is_new=b in NEW_BOTTLENECK_NAMES,
                         runtime_s=info.get("runtime_s"), run_info=info,
                         kind=("new" if b in NEW_BOTTLENECK_NAMES else "paper"))

    if globals().get("INCLUDE_PRETRAINED", False):
        for pth in sorted((REPO / "trained_models").glob("Branch_Encoder_Decoder_*/*_Model.pth")):
            b = pth.parent.name[len("Branch_Encoder_Decoder_"):]
            mid = "pretrained_%s" % b
            runs[mid] = dict(model_id=mid, bottleneck=b, trained_on="AUTHORS", seed="",
                             path=pth, folder=pth.parent, epochs=None, is_new=False,
                             runtime_s=None, run_info={}, kind="pretrained")
    return dict(sorted(runs.items()))


print("helpers ready")

helpers ready


## Part 1: self-test

Switch `RUN_SELFTEST`. Seconds on CPU, trains nothing. Run it before spending a
GPU session.

Five checks:

1. **Parameter counts.** The five paper bottlenecks must reproduce the counts
   measured on the earlier Kaggle runs exactly (78,090 / 86,410 / 178,442 /
   111,370 / 640,394). `attention_PE` must equal `attention`.
2. **Forward shapes.** `(b, 3, 600)` in, two `(b, 3, 600)` out, for all nine.
3. **Batch independence.** A row scored alone must give the same answer as the
   same row inside a batch. Only the paper's `Transformer` is expected to fail,
   because `TransformerEncoderLayer` is built without `batch_first`, defaults to
   `False`, and so reads `(batch, steps, features)` as `(seq, batch, feature)` --
   it attends across the batch instead of along time.
4. **Positional encoding.** `attention` must be exactly insensitive to reversing
   the time axis (self-attention without positional information is permutation
   invariant); `attention_PE` must not be.
5. **EV implementation** must match `sklearn.explained_variance_score`.

In [5]:
if RUN_SELFTEST:
    import copy
    import torch
    from autoencoder_1D_models_torch import (SeisSeparator, SeismogramDecoder,
                                             SeismogramEncoder)
    from wdn_model_ext import (AttentionPEBottleneck, NEW_BOTTLENECKS,
                               make_bottlenecks)

    def paper_bottlenecks(name):
        """Mirror of step3's construction for the paper's five names."""
        if name == "None":
            b = None
        elif name == "Linear":
            b = torch.nn.Linear(64, 64, dtype=torch.float64)
        elif name == "LSTM":
            b = torch.nn.LSTM(64, 32, 2, bidirectional=True, batch_first=True,
                              dtype=torch.float64)
        elif name == "attention":
            b = torch.nn.MultiheadAttention(64, 4, 0.2, batch_first=True,
                                            dtype=torch.float64)
        elif name == "Transformer":
            layer = torch.nn.TransformerEncoderLayer(d_model=64, nhead=4,
                                                     dtype=torch.float64)
            b = torch.nn.TransformerEncoder(layer, num_layers=1)
        else:
            raise NameError(name)
        return (None, None) if b is None else (b, copy.deepcopy(b))

    def build(name):
        if name in PAPER_BOTTLENECK_NAMES:
            b1, b2 = paper_bottlenecks(name)
        else:
            b1, b2 = make_bottlenecks(name)
        return SeisSeparator("Branch_Encoder_Decoder_" + name, SeismogramEncoder(),
                             SeismogramDecoder(bottleneck=b1),
                             SeismogramDecoder(bottleneck=b2))

    torch.manual_seed(0)
    ALL_NAMES = list(PAPER_BOTTLENECK_NAMES) + list(NEW_BOTTLENECK_NAMES)
    failures, models, prows = [], {}, []

    print("=" * 76)
    print("1. PARAMETER COUNTS")
    print("=" * 76)
    print("%-20s %12s %14s  %s" % ("bottleneck", "params", "vs measured", "kind"))
    for nm in ALL_NAMES:
        m = build(nm)
        models[nm] = m
        p = n_params(m)
        if nm in PAPER_PARAM_REF:
            d = p - PAPER_PARAM_REF[nm]
            flag = "OK" if d == 0 else "MISMATCH %+d" % d
            if d:
                failures.append("params %s %+d" % (nm, d))
        else:
            flag = "-"
        print("%-20s %12d %14s  %s" % (nm, p, flag, "new" if nm in NEW_BOTTLENECKS else "paper"))
        prows.append(dict(bottleneck=nm, params=p,
                          kind="new" if nm in NEW_BOTTLENECKS else "paper",
                          params_measured_phase1=PAPER_PARAM_REF.get(nm, "")))
    if n_params(models["attention_PE"]) != n_params(models["attention"]):
        failures.append("attention_PE parameter count differs from attention")
    print("\nattention_PE == attention parameter count: %s"
          % ("yes" if n_params(models["attention_PE"]) == n_params(models["attention"]) else "NO"))

    print("\n" + "=" * 76)
    print("2. FORWARD SHAPES   input (8, 3, 600) float64")
    print("=" * 76)
    x = torch.randn(8, 3, 600, dtype=torch.float64)
    for nm in ALL_NAMES:
        models[nm].eval()
        with torch.no_grad():
            eq, nz = models[nm](x)
        ok = tuple(eq.shape) == (8, 3, 600) and tuple(nz.shape) == (8, 3, 600)
        print("%-20s eq %-14s noise %-14s %s"
              % (nm, tuple(eq.shape), tuple(nz.shape), "OK" if ok else "BAD"))
        if not ok:
            failures.append("shape %s" % nm)

    print("\n" + "=" * 76)
    print("3. BATCH INDEPENDENCE   row 0 alone vs row 0 inside a batch of 8 (eval mode)")
    print("=" * 76)
    print("%-20s %14s  %s" % ("bottleneck", "max |diff|", "verdict"))
    for nm in ALL_NAMES:
        models[nm].eval()
        with torch.no_grad():
            eq_b, _ = models[nm](x)
            eq_s, _ = models[nm](x[:1])
        d = float((eq_b[:1] - eq_s).abs().max())
        dep = d >= 1e-10
        print("%-20s %14.3e  %s" % (nm, d, "*** BATCH-DEPENDENT ***" if dep else "batch-independent"))
        for r in prows:
            if r["bottleneck"] == nm:
                r["batch_dep_max_abs_diff"] = d
        if dep and nm != "Transformer":
            failures.append("batch dependence %s (%.3e)" % (nm, d))
    print("\nOnly 'Transformer' is expected to be batch-dependent.")

    print("\n" + "=" * 76)
    print("4. POSITIONAL ENCODING")
    print("=" * 76)
    torch.manual_seed(0)
    bare = torch.nn.MultiheadAttention(64, 4, 0.0, batch_first=True,
                                       dtype=torch.float64).eval()
    torch.manual_seed(0)
    withpe = AttentionPEBottleneck(64, 4, 0.0).eval()
    z = torch.randn(2, 75, 64, dtype=torch.float64)
    with torch.no_grad():
        b1, _ = bare(z, z, z)
        b2, _ = bare(z.flip(1), z.flip(1), z.flip(1))
        p1, p2 = withpe(z), withpe(z.flip(1))
    bare_gap = float((b1.flip(1) - b2).abs().max())
    pe_gap = float((p1.flip(1) - p2).abs().max())
    print("time-reversal sensitivity, attention      : %.3e   (0 => order-blind)" % bare_gap)
    print("time-reversal sensitivity, attention + PE : %.3e   (>0 => order-aware)" % pe_gap)
    if pe_gap <= bare_gap:
        failures.append("positional encoding adds no order sensitivity")

    print("\n" + "=" * 76)
    print("5. EV MATCHES SCIKIT-LEARN")
    print("=" * 76)
    from sklearn.metrics import explained_variance_score
    rng = np.random.default_rng(1)
    yt = rng.normal(size=(40, 600, 3))
    yp = yt + 0.3 * rng.normal(size=yt.shape)
    yt[3, :, 1] = 7.0                      # zero-variance target, numerator != 0
    yt[4, :, 2] = 2.0                      # zero-variance target ...
    yp[4, :, 2] = 2.0                      # ... and exact prediction
    mine = ev_raw(yt, yp)
    ref = np.array([[explained_variance_score(yt[i, :, c], yp[i, :, c])
                     for c in range(3)] for i in range(len(yt))])
    gap = float(np.abs(mine - ref).max())
    print("max |ours - sklearn| over %d row-components = %.3e" % (mine.size, gap))
    print("zero-variance rows handled: row3 %s  row4 %s"
          % (np.round(mine[3], 4).tolist(), np.round(mine[4], 4).tolist()))
    if gap > 1e-9:
        failures.append("EV does not match sklearn (%.3e)" % gap)

    pd.DataFrame(prows).to_csv(RESULTS / "table_variant_params.csv", index=False)
    print("\nwrote", RESULTS / "table_variant_params.csv")

    print("\n" + "=" * 76)
    if failures:
        print("FAILURES (%d):" % len(failures))
        for f in failures:
            print("  - " + f)
        raise AssertionError("self-test failed: %s" % failures)
    print("ALL CHECKS PASSED")
    log("Part 1 self-test passed for %d bottlenecks" % len(ALL_NAMES))

1. PARAMETER COUNTS
bottleneck                 params    vs measured  kind
None                        78090             OK  paper
Linear                      86410             OK  paper
LSTM                       178442             OK  paper
attention                  111370             OK  paper
Transformer                640394             OK  paper
attention_PE               111370              -  new
attention_LSTM             161546              -  new
hybrid                     144906              -  new
Transformer_fixed          178058              -  new

attention_PE == attention parameter count: yes

2. FORWARD SHAPES   input (8, 3, 600) float64


/tmp/ipykernel_23/4284140795.py:24: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  b = torch.nn.TransformerEncoder(layer, num_layers=1)


None                 eq (8, 3, 600)    noise (8, 3, 600)    OK
Linear               eq (8, 3, 600)    noise (8, 3, 600)    OK
LSTM                 eq (8, 3, 600)    noise (8, 3, 600)    OK
attention            eq (8, 3, 600)    noise (8, 3, 600)    OK
Transformer          eq (8, 3, 600)    noise (8, 3, 600)    OK
attention_PE         eq (8, 3, 600)    noise (8, 3, 600)    OK
attention_LSTM       eq (8, 3, 600)    noise (8, 3, 600)    OK
hybrid               eq (8, 3, 600)    noise (8, 3, 600)    OK
Transformer_fixed    eq (8, 3, 600)    noise (8, 3, 600)    OK

3. BATCH INDEPENDENCE   row 0 alone vs row 0 inside a batch of 8 (eval mode)
bottleneck               max |diff|  verdict
None                      0.000e+00  batch-independent
Linear                    0.000e+00  batch-independent
LSTM                      2.776e-17  batch-independent
attention                 0.000e+00  batch-independent
Transformer               1.158e-04  *** BATCH-DEPENDENT ***
attention_PE              0.0

## Part 2: the paper's `Transformer` bottleneck depends on the evaluation batch size

Switch `RUN_BATCHDEP`. CPU, trains nothing, needs the already-trained paper
models. This is the cheapest new result in the notebook and the one Part 1
predicts structurally.

`step3` builds the Transformer bottleneck as
`TransformerEncoderLayer(d_model=64, nhead=4)` and passes no `batch_first`, so
it defaults to `False`. `SeismogramDecoder.forward` permutes the latent to
`(batch, steps, features)` before the bottleneck, so the encoder reads a
`(128, 75, 64)` tensor as sequence length 128, batch 75 -- it attends across the
128 traces in the batch and never along the 75 time steps.

Two measurements:

- pooled median EV as a function of the evaluation batch size, for the
  `Transformer` bottleneck and for `attention` as a control;
- the same at a fixed batch size of 128, under `BATCHDEP_PERMS` different row
  orderings, which changes only which traces share a batch.

A correct model is flat in both. Whatever spread appears for the Transformer is
the size of the artefact in the EV numbers reported for it, and it means the
evaluation batch size has to be stated alongside them.

In [6]:
if RUN_BATCHDEP:
    runs_all = discover_runs()
    print("runs found:", list(runs_all))

    bd_set, bd_prefix = None, None
    for cand, pref in (("STEAD", "stead"), ("INSTANCE", "instance")):
        if any(k.startswith(pref + "_") for k in runs_all):
            fn = STEAD_FILE if cand == "STEAD" else INSTANCE_FILE
            if find_one(fn, required=False) is not None:
                bd_set, bd_prefix = cand, pref
                break
    if bd_set is None:
        raise RuntimeError("no trained runs with a matching test file; attach "
                           "wdn-runs-seed99 + wdn-training-data (or the INSTANCE pair)")
    print("batch-dependence test on the %s test split" % bd_set)

    # ------------------------------------------------- test rows, evenly spaced
    if bd_set == "STEAD":
        path = find_one(STEAD_FILE)
        with h5py.File(path, "r") as f:
            n_all = f["X_train"].shape[0]
        _, _, te = stead_split_indices(n_all)
    else:
        path = find_one(INSTANCE_FILE)
        with h5py.File(path, "r") as f:
            te = np.where(f["split_id"][:] == 2)[0]
    te = np.sort(te)
    pick = te[np.linspace(0, len(te) - 1, min(BATCHDEP_ROWS, len(te))).astype(int)]
    print("test rows available %d, using %d" % (len(te), len(pick)))
    _, Xb, Yb = load_rows(path, pick)
    print("loaded", Xb.shape, Xb.dtype)

    bd_rows, perm_rows, spread_rows = [], [], []
    for b in BATCHDEP_MODELS:
        mid = model_id(b, bd_set, 99)
        if mid not in runs_all:
            print("skip %s (no run)" % mid)
            continue
        model = load_model(runs_all[mid]["path"])
        pf = make_predictor("model", model)
        print("\n--- %s (%d params) ---" % (mid, n_params(model)))

        # (a) batch-size sweep
        for bs in BATCHDEP_SIZES:
            if bs > len(Xb):
                continue
            t0 = time.time()
            r = eval_predictor(pf, Xb, Yb, batch=bs)
            med_eq = float(np.median(r["ev_eq"]))
            med_nz = float(np.median(r["ev_noise"]))
            loss = float((r["mse_eq"] + r["mse_noise"]).mean())
            bd_rows.append(dict(model_id=mid, bottleneck=b, test_set=bd_set,
                                n_rows=len(Xb), batch_size=bs,
                                median_ev_eq=med_eq, median_ev_noise=med_nz,
                                mean_clipped_ev_eq=float(r["ev_eq_clipped"].mean()),
                                test_loss=loss, seconds=round(time.time() - t0, 1)))
            print("  batch %5d  median EV eq %.4f  noise %.4f  loss %.4f  (%.0fs)"
                  % (bs, med_eq, med_nz, loss, time.time() - t0))

        # (b) row-order sweep at batch size 128
        base_bs = 128 if 128 <= len(Xb) else len(Xb)
        rng = np.random.default_rng(BOOT_SEED)
        per_row = []
        for k in range(BATCHDEP_PERMS):
            p = np.arange(len(Xb)) if k == 0 else rng.permutation(len(Xb))
            r = eval_predictor(pf, Xb[p], Yb[p], batch=base_bs)
            inv = np.argsort(p)
            per_row.append(r["ev_eq"][inv].mean(axis=1))
            med_eq = float(np.median(r["ev_eq"]))
            perm_rows.append(dict(model_id=mid, bottleneck=b, test_set=bd_set,
                                  batch_size=base_bs, permutation=k,
                                  ordering="file order" if k == 0 else "shuffled",
                                  median_ev_eq=med_eq))
            print("  perm %d (batch %d)  median EV eq %.4f" % (k, base_bs, med_eq))
        per_row = np.vstack(per_row)
        sp = per_row.max(axis=0) - per_row.min(axis=0)
        spread_rows.append(dict(model_id=mid, bottleneck=b, test_set=bd_set,
                                n_perms=BATCHDEP_PERMS, batch_size=base_bs,
                                median_ev_spread_across_perms=float(
                                    np.ptp([r["median_ev_eq"] for r in perm_rows
                                            if r["model_id"] == mid])),
                                per_row_spread_mean=float(sp.mean()),
                                per_row_spread_p95=float(np.percentile(sp, 95)),
                                per_row_spread_max=float(sp.max())))
        print("  per-row EV spread across orderings: mean %.2e  p95 %.2e  max %.2e"
              % (sp.mean(), np.percentile(sp, 95), sp.max()))
        del model

    pd.DataFrame(bd_rows).to_csv(RESULTS / "table_batchdep.csv", index=False)
    pd.DataFrame(perm_rows).to_csv(RESULTS / "table_batchdep_perm.csv", index=False)
    pd.DataFrame(spread_rows).to_csv(RESULTS / "table_batchdep_spread.csv", index=False)
    print("\nwrote table_batchdep.csv, table_batchdep_perm.csv, table_batchdep_spread.csv")
    log("Part 2 batch-dependence on %s: %d models, %d rows"
        % (bd_set, len(set(r["model_id"] for r in bd_rows)), len(Xb)))
    del Xb, Yb

runs found: ['instance_LSTM_seed99', 'instance_Linear_seed99', 'instance_None_seed99', 'instance_Transformer_seed99', 'instance_attention_seed99', 'pretrained_LSTM', 'pretrained_Linear', 'pretrained_None', 'pretrained_Transformer', 'pretrained_attention', 'stead_LSTM_seed99', 'stead_Linear_seed99', 'stead_None_seed99', 'stead_Transformer_fixed_seed99', 'stead_Transformer_seed99', 'stead_attention_LSTM_seed99', 'stead_attention_PE_seed99', 'stead_attention_seed99', 'stead_hybrid_seed99']
batch-dependence test on the STEAD test split
test rows available 37400, using 2048
loaded (2048, 600, 3) float64

--- stead_Transformer_seed99 (640394 params) ---
  batch     1  median EV eq 0.9537  noise 0.6198  loss 0.2546  (15s)
  batch     2  median EV eq 0.9445  noise 0.7344  loss 0.1617  (10s)
  batch     8  median EV eq 0.9471  noise 0.8579  loss 0.0898  (11s)
  batch    32  median EV eq 0.9536  noise 0.8894  loss 0.0710  (10s)
  batch   128  median EV eq 0.9556  noise 0.8955  loss 0.0669  (10s)

## Part 3: training, one variant per session

Switches `RUN_SMOKE` then `RUN_TRAIN`. Needs a **GPU** session. Set `VARIANT`
and `TRAIN_ON` in the switches cell.

Training is the user's own patched `step3`, copied and given the three edits in
Part 0. Nothing else changes: same loss (`MSELoss` on both branches), optimiser
(Adam, lr 1e-3), warm-up-then-decay schedule, batch 128, 100 epochs, no early
stopping, float64, same split seeds. Only the bottleneck module is new, so the
result lands in the same comparison as the paper's five.

Run `RUN_SMOKE` first: 2 epochs on `SMOKE_SUBSET` rows, which prints a projected
full runtime. Projections from the Phase 1 STEAD runs (`attention` 12,788 s,
`LSTM` 15,686 s) put `attention_PE` near 3.6 h, `Transformer_fixed` near 4 h,
`hybrid` near 4 h and `attention_LSTM` near 4.5 h.

**Recommended:** train on **STEAD**. The INSTANCE training split gives 141
optimiser updates per epoch from about 735 distinct earthquake traces, where an
architecture change has little room to show itself; STEAD gives 877 updates per
epoch over 112,200 rows.

In [7]:
if RUN_SMOKE or RUN_TRAIN:
    print("data path configuration seen by step3 (from config.py):")
    _cfgp = REPO / "config.py"
    if _cfgp.exists():
        for _l in _cfgp.read_text().split("\n"):
            if re.search(r"hdf5|\.h5|data_dir|training_datasets|model_dataset", _l, re.I):
                print("   ", _l.strip()[:150])
    else:
        print("    config.py not present next to step3")
    print("\nstep3 flags available:", STEP3_FLAGS)
    _data_flag = next((f for f in ("data", "data_file", "dataset", "input", "hdf5")
                       if f in STEP3_FLAGS), None)
    _target = STEAD_FILE if TRAIN_ON == "STEAD" else INSTANCE_FILE
    _target_path = find_one(_target, required=False)
    print("target data file:", _target_path)
    if _data_flag is None and _target_path is not None:
        print("NOTE: step3 has no data-path flag, so it uses the path in config.py.\n"
              "      Confirm that path resolves to %s before a full run." % _target)

    def run_step3(bottleneck, seed, epochs, subset=None, tag=None, log_name="train.log"):
        cmd = [sys.executable, "-u", str(STEP3.name), "--bottleneck", bottleneck]
        if "seed" in STEP3_FLAGS:
            cmd += ["--seed", str(seed)]
        if "epochs" in STEP3_FLAGS:
            cmd += ["--epochs", str(epochs)]
        if subset is not None and "subset" in STEP3_FLAGS:
            cmd += ["--subset", str(subset)]
        if tag is not None and "tag" in STEP3_FLAGS:
            cmd += ["--tag", str(tag)]
        if TRAIN_ON == "INSTANCE" and "presplit" in STEP3_FLAGS:
            cmd += ["--presplit"]
        if _data_flag is not None and _target_path is not None:
            cmd += ["--" + _data_flag, str(_target_path)]
        print("$", " ".join(cmd), "\n  (cwd=%s)" % REPO)
        t0 = time.time()
        lp = LOGS / log_name
        with open(lp, "a") as lf:
            lf.write("\n=== %s %s ===\n$ %s\n"
                     % (datetime.datetime.now().isoformat(timespec="seconds"),
                        bottleneck, " ".join(cmd)))
            proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in proc.stdout:
                lf.write(line)
                sys.stdout.write(line)
            proc.wait()
        dt = time.time() - t0
        print("\nexit code %s after %.1f s (%.2f h); log: %s" % (proc.returncode, dt, dt / 3600, lp))
        if proc.returncode != 0:
            raise RuntimeError("step3 failed with exit code %s; see %s" % (proc.returncode, lp))
        return dt

    def newest_run(bottleneck):
        cands = [p for p in find_all("*_Model.pth", roots=[REPO, WORK, TEMP])
                 if bottleneck in p.parent.name]
        return max(cands, key=lambda p: p.stat().st_mtime) if cands else None

if RUN_SMOKE:
    print("=" * 76)
    print("SMOKE: %s, %d epochs, subset %s" % (VARIANT, SMOKE_EPOCHS, SMOKE_SUBSET))
    print("=" * 76)
    if "subset" not in STEP3_FLAGS:
        print("WARNING: step3 has no --subset flag; the smoke run will use the FULL data.")
    if "epochs" not in STEP3_FLAGS:
        print("WARNING: step3 has no --epochs flag, so SMOKE_EPOCHS cannot be passed and\n"
              "         this 'smoke' run will train for step3's own epoch count (100).\n"
              "         Stop here and tell me step3's flag names: %s" % STEP3_FLAGS)
    dt = run_step3(VARIANT, SEED, SMOKE_EPOCHS, subset=SMOKE_SUBSET,
                   tag="smoke", log_name="smoke.log")
    n_train = (112200 if TRAIN_ON == "STEAD" else 18000)
    scale = (n_train / SMOKE_SUBSET) if "subset" in STEP3_FLAGS else 1.0
    per_epoch = dt / max(SMOKE_EPOCHS, 1)
    print("\nprojection: %.1f s/epoch on the subset -> about %.2f h for %d epochs "
          "on the full %s training split"
          % (per_epoch, per_epoch * scale * EPOCHS / 3600, EPOCHS, TRAIN_ON))
    log("Part 3 smoke %s on %s: %.1f s for %d epochs" % (VARIANT, TRAIN_ON, dt, SMOKE_EPOCHS))

if RUN_TRAIN:
    print("=" * 76)
    print("FULL TRAINING: %s on %s, seed %d, %d epochs" % (VARIANT, TRAIN_ON, SEED, EPOCHS))
    print("=" * 76)
    dt = run_step3(VARIANT, SEED, EPOCHS, log_name="train_%s.log" % VARIANT)
    mp = newest_run(VARIANT)
    print("\nmodel:", mp)
    if mp is not None:
        hist = sorted(mp.parent.glob("*_Training_history.hdf5"))
        if hist:
            with h5py.File(hist[0], "r") as f:
                tr, va = f["loss"][:], f["val_loss"][:]
            best = int(np.argmin(va))
            print("epochs run        %d" % len(tr))
            print("final train loss  %.4f" % tr[-1])
            print("final val loss    %.4f" % va[-1])
            print("best val loss     %.4f at epoch %d (of %d)" % (va[best], best + 1, len(va)))
            if best < len(va) - 1:
                print("NOTE: validation loss was lower at epoch %d than at the last epoch, "
                      "but step3 saves the last epoch." % (best + 1))
        print("\nFolder contents:")
        for p in sorted(mp.parent.iterdir()):
            print("   %-58s %8.1f KB" % (p.name, p.stat().st_size / 1024))
        print("\nDownload this folder, then add it to the runs dataset for the "
              "evaluation session.")
    log("Part 3 full %s on %s seed %d: %.1f s" % (VARIANT, TRAIN_ON, SEED, dt))

## Part 4: evaluation on the test splits

Switch `RUN_EVAL`. CPU is enough. Scores every finished run it can find, plus
the identity and band-pass baselines, on each split named in `EVAL_ON`.

Model ids follow the Phase 2 convention, so the new rows sit beside the existing
ones: `stead_<bottleneck>_seed99` for a STEAD-trained model,
`instance_<bottleneck>_seed99` for an INSTANCE-trained one. A STEAD-trained model
scored on the STEAD test split is the E1 comparison; on the INSTANCE test split
it is E2; an INSTANCE-trained model on the INSTANCE test split is E3.

Outputs in `results/`:

| File | Content |
|---|---|
| `table_variants_eval.csv` | one row per model x test split: params, pooled median EV (earthquake, noise) with 95% CI, mean clipped EV with CI, fraction of EV < 0, test loss with CI, mean correlation, consistency residual, runtime |
| `table_variants_paired.csv` | paired mean differences of per-row clipped EV, bootstrap CI, `significant_at_95` |
| `table_variants_ev_vs_snr.csv` | mean clipped EV in 12 log-spaced bins of linear input SNR |
| `per_sample_<model_id>__<split>.csv.gz` | per-row EV, MSE, correlation, residual and metadata |
| `examples_<split>.npz` | input, true earthquake and predictions for three example rows |

The `attention_PE` vs `attention` row of the paired table is the cleanest
comparison in the whole notebook: identical parameter count, identical training
recipe, one structural difference.

In [8]:
SNR_EDGES = np.logspace(-1, 2, 13)     # 12 log-spaced bins, linear SNR 0.1 to 100

PAIR_SPECS = [("attention_PE", "attention"), ("Transformer_fixed", "Transformer"),
              ("attention_LSTM", "attention"), ("attention_LSTM", "LSTM"),
              ("hybrid", "LSTM"), ("hybrid", "attention"),
              ("attention_PE", "LSTM"), ("Transformer_fixed", "LSTM"),
              ("attention", "LSTM")]


def load_test_set(which):
    """Test rows, targets, per-row metadata and bootstrap cluster labels."""
    if which == "STEAD":
        path = find_one(STEAD_FILE)
        with h5py.File(path, "r") as f:
            n_all = f["X_train"].shape[0]
            src_all = f["source_index"][:]
            n_poha = int(f.attrs.get("n_poha", 87000))
        _, _, te = stead_split_indices(n_all)
        te = np.sort(te)
        if MAX_TEST_ROWS:
            te = te[np.linspace(0, len(te) - 1, min(MAX_TEST_ROWS, len(te))).astype(int)]
        rows, X, Y = load_rows(path, te)
        src = src_all[rows]
        meta = pd.DataFrame(dict(file_row=rows, source_index=src))
        meta["source"] = np.where(src < n_poha, "POHA", "STEAD")
        for col in ("snr", "shift", "quake_energy_frac", "earthquake_name"):
            meta[col] = np.nan if col != "earthquake_name" else ""
        pp, qp = find_one(STEAD_POHA, required=False), find_one(STEAD_QUAKE, required=False)
        if pp is not None and qp is not None:
            def grab(p):
                with h5py.File(p, "r") as f:
                    d = {}
                    for col in ("snr", "shift", "quake_energy_frac"):
                        d[col] = f[col][:] if col in f else None
                    nm = f["earthquake_name"][:] if "earthquake_name" in f else None
                    d["earthquake_name"] = (
                        np.array([s.decode() if isinstance(s, bytes) else str(s) for s in nm])
                        if nm is not None else None)
                return d
            P, Q = grab(pp), grab(qp)
            is_p = src < n_poha
            ip, iq = src[is_p], src[~is_p] - n_poha
            for col in ("snr", "shift", "quake_energy_frac"):
                if P[col] is not None and Q[col] is not None:
                    v = np.full(len(src), np.nan)
                    v[is_p] = P[col][ip]
                    v[~is_p] = Q[col][iq]
                    meta[col] = v
            if P["earthquake_name"] is not None and Q["earthquake_name"] is not None:
                nm = np.empty(len(src), dtype=object)
                nm[is_p] = P["earthquake_name"][ip]
                nm[~is_p] = Q["earthquake_name"][iq]
                meta["earthquake_name"] = nm
                clusters = np.array(["%s:%s" % (s, n) for s, n in
                                     zip(meta["source"], meta["earthquake_name"])])
            else:
                clusters = src.astype(str)
        else:
            print("WARNING: intermediate STEAD files not attached -- no per-row SNR, "
                  "and the bootstrap clusters fall back to source_index "
                  "(which under-counts correlation between rows of one earthquake).")
            clusters = src.astype(str)
    else:
        path = find_one(INSTANCE_FILE)
        with h5py.File(path, "r") as f:
            te = np.where(f["split_id"][:] == 2)[0]
            te = np.sort(te)
            if MAX_TEST_ROWS:
                te = te[np.linspace(0, len(te) - 1, min(MAX_TEST_ROWS, len(te))).astype(int)]
            cols = {}
            for col in ("snr", "shift", "quake_energy_frac", "source_id",
                        "source_magnitude", "event_snr_mean_db", "p_time_in_window_s",
                        "station_channels", "earthquake_name"):
                if col in f:
                    v = f[col][:][te]
                    if v.dtype.kind in "SO":
                        v = np.array([s.decode() if isinstance(s, bytes) else str(s) for s in v])
                    cols[col] = v
        rows, X, Y = load_rows(path, te)
        cols["file_row"] = rows
        meta = pd.DataFrame(cols)
        clusters = (meta["source_id"].astype(str).values if "source_id" in meta
                    else rows.astype(str))
    print("%s test split: %d rows, X %s" % (which, len(X), X.shape))
    print("  distinct bootstrap clusters:", len(np.unique(clusters)))
    return dict(name=which, path=path, rows=rows, X=X, Y=Y, meta=meta, clusters=clusters)


if RUN_EVAL:
    runs = discover_runs()
    print("\nruns discovered (%d):" % len(runs))
    for k, v in runs.items():
        print("  %-34s %-9s epochs=%s params-at-load=?" % (k, v["trained_on"], v["epochs"]))
    if not runs:
        raise RuntimeError("no finished runs found; attach the run datasets")

    summary_rows, paired_rows, snr_rows = [], [], []

    for split_name in EVAL_ON:
        fn = STEAD_FILE if split_name == "STEAD" else INSTANCE_FILE
        if find_one(fn, required=False) is None:
            print("\nskipping %s split: %s not attached" % (split_name, fn))
            continue
        print("\n" + "=" * 76)
        print("TEST SPLIT: %s" % split_name)
        print("=" * 76)
        TS = load_test_set(split_name)
        X, Y, meta = TS["X"], TS["Y"], TS["meta"]
        boot = make_boot_indices(TS["clusters"])

        to_score = [(mid, "model", info) for mid, info in runs.items()]
        to_score += [("baseline_identity", "identity", None),
                     ("baseline_bandpass", "bandpass", None)]

        per_model = {}
        for mid, kind, info in to_score:
            t0 = time.time()
            if kind == "model":
                model = load_model(info["path"])
                npar = n_params(model)
                pf = make_predictor("model", model)
            else:
                npar = 0
                pf = make_predictor(kind)
            r = eval_predictor(pf, X, Y)
            per_model[mid] = r

            med_eq = lambda ix, _r=r: float(np.median(_r["ev_eq"][ix]))
            med_nz = lambda ix, _r=r: float(np.median(_r["ev_noise"][ix]))
            cl_eq = lambda ix, _r=r: float(_r["ev_eq_clipped"][ix].mean())
            cl_nz = lambda ix, _r=r: float(_r["ev_noise_clipped"][ix].mean())
            loss_f = lambda ix, _r=r: float((_r["mse_eq"][ix] + _r["mse_noise"][ix]).mean())

            ci = {k: boot_ci(f, boot) for k, f in
                  (("med_eq", med_eq), ("med_nz", med_nz), ("cl_eq", cl_eq),
                   ("cl_nz", cl_nz), ("loss", loss_f))}
            allrows = np.arange(len(X))
            row = dict(
                model_id=mid, test_split=split_name,
                kind=(info["kind"] if info else "baseline"),
                bottleneck=(info["bottleneck"] if info else ""),
                trained_on=(info["trained_on"] if info else ""),
                seed=(info["seed"] if info else ""),
                n_params=npar, n_rows=len(X), eval_batch=EVAL_BATCH,
                median_ev_eq=med_eq(allrows),
                median_ev_eq_lo=ci["med_eq"][0], median_ev_eq_hi=ci["med_eq"][1],
                median_ev_noise=med_nz(allrows),
                median_ev_noise_lo=ci["med_nz"][0], median_ev_noise_hi=ci["med_nz"][1],
                mean_clipped_ev_eq=cl_eq(allrows),
                mean_clipped_ev_eq_lo=ci["cl_eq"][0], mean_clipped_ev_eq_hi=ci["cl_eq"][1],
                mean_clipped_ev_noise=cl_nz(allrows),
                mean_clipped_ev_noise_lo=ci["cl_nz"][0], mean_clipped_ev_noise_hi=ci["cl_nz"][1],
                frac_ev_eq_lt0=float((r["ev_eq"] < 0).mean()),
                frac_ev_noise_lt0=float((r["ev_noise"] < 0).mean()),
                test_loss=loss_f(allrows),
                test_loss_lo=ci["loss"][0], test_loss_hi=ci["loss"][1],
                mean_corr_eq=float(r["corr_eq"].mean()),
                consistency_residual=float(np.median(r["resid"])),
                runtime_s=(info.get("runtime_s") if info else ""),
                eval_seconds=round(time.time() - t0, 1))
            summary_rows.append(row)
            print("  %-34s medEV eq %.4f  noise %.4f  loss %.4f  resid %.2e  (%.0fs)"
                  % (mid, row["median_ev_eq"], row["median_ev_noise"],
                     row["test_loss"], row["consistency_residual"], row["eval_seconds"]))

            out = meta.copy()
            for c in range(r["ev_eq"].shape[1]):
                out["ev_eq_c%d" % c] = r["ev_eq"][:, c]
                out["ev_noise_c%d" % c] = r["ev_noise"][:, c]
            out["ev_eq_mean_clipped"] = r["ev_eq_clipped"]
            out["ev_noise_mean_clipped"] = r["ev_noise_clipped"]
            for c in ("mse_eq", "mse_noise", "corr_eq", "resid"):
                out[c] = r[c]
            out.to_csv(RESULTS / ("per_sample_%s__%s.csv.gz" % (mid, split_name)),
                       index=False, compression="gzip")
            if kind == "model":
                del model
            del pf

            if "snr" in meta and np.isfinite(pd.to_numeric(meta["snr"], errors="coerce")).any():
                s = pd.to_numeric(meta["snr"], errors="coerce").values
                which_bin = np.digitize(s, SNR_EDGES) - 1
                for b in range(len(SNR_EDGES) - 1):
                    ix = np.where(which_bin == b)[0]
                    if len(ix) < 5:
                        continue
                    snr_rows.append(dict(
                        model_id=mid, test_split=split_name, snr_bin=b,
                        snr_lo=SNR_EDGES[b], snr_hi=SNR_EDGES[b + 1], n=len(ix),
                        mean_clipped_ev_eq=float(r["ev_eq_clipped"][ix].mean()),
                        mean_clipped_ev_noise=float(r["ev_noise_clipped"][ix].mean())))

        # ---------------------------------------------- paired comparisons
        for a_b, b_b in PAIR_SPECS:
            for pref in ("stead", "instance"):
                ida, idb = ("%s_%s_seed99" % (pref, a_b)), ("%s_%s_seed99" % (pref, b_b))
                if ida not in per_model or idb not in per_model:
                    continue
                for branch, key in (("earthquake", "ev_eq_clipped"), ("noise", "ev_noise_clipped")):
                    d = per_model[ida][key] - per_model[idb][key]
                    lo, hi = boot_ci(lambda ix, _d=d: float(_d[ix].mean()), boot)
                    paired_rows.append(dict(
                        test_split=split_name, trained_on=pref.upper(), branch=branch,
                        model_a=ida, model_b=idb, metric="mean_clipped_ev",
                        mean_diff=float(d.mean()), ci_lo=lo, ci_hi=hi,
                        significant_at_95=bool(lo > 0 or hi < 0)))
                    print("  paired %-22s %-10s %-32s - %-32s  %+.4f [%+.4f, %+.4f] %s"
                          % (branch, pref, a_b, b_b, d.mean(), lo, hi,
                             "*" if (lo > 0 or hi < 0) else ""))

        # ---------------------------------------------- example waveforms
        ref = None
        for cand in sorted([m for m in per_model if any(nb in m for nb in NEW_BOTTLENECK_NAMES)],
                           key=lambda m: -float(np.median(per_model[m]["ev_eq"]))):
            ref = cand
            break
        if ref is None:
            ref = max([m for m in per_model if m.startswith(("stead_", "instance_"))],
                      key=lambda m: float(np.median(per_model[m]["ev_eq"])), default=None)
        if ref is not None:
            q = per_model[ref]["ev_eq_clipped"]
            picks = [int(np.argsort(q)[int(f * (len(q) - 1))]) for f in (0.95, 0.50, 0.05)]
            labels = ["good (95th pct)", "median", "failure (5th pct)"]
            store = dict(rows=np.array(picks), labels=np.array(labels),
                         x=X[picks], y=Y[picks], reference=ref,
                         ev_ref=q[picks], eval_batch=EVAL_BATCH)
            shown = [ref]
            base = ref.split("_")[0]
            for pb in ("attention", "LSTM"):
                cid = "%s_%s_seed99" % (base, pb)
                if cid in per_model and cid != ref:
                    shown.append(cid)
                    break
            for mid in shown:
                if mid in runs:
                    m = load_model(runs[mid]["path"])
                    pf = make_predictor("model", m)
                    peq, pnz = pf(X[picks])
                    store["pred_eq__" + mid] = peq
                    store["pred_noise__" + mid] = pnz
                    del m, pf
            store["shown"] = np.array(shown)
            np.savez_compressed(RESULTS / ("examples_%s.npz" % split_name), **store)
            print("\n  examples_%s.npz: rows %s, reference %s, models %s"
                  % (split_name, picks, ref, shown))

        del TS, X, Y, per_model, boot

    pd.DataFrame(summary_rows).to_csv(RESULTS / "table_variants_eval.csv", index=False)
    pd.DataFrame(paired_rows).to_csv(RESULTS / "table_variants_paired.csv", index=False)
    pd.DataFrame(snr_rows).to_csv(RESULTS / "table_variants_ev_vs_snr.csv", index=False)
    print("\nwrote table_variants_eval.csv (%d rows), table_variants_paired.csv (%d), "
          "table_variants_ev_vs_snr.csv (%d)"
          % (len(summary_rows), len(paired_rows), len(snr_rows)))
    log("Part 4 eval: %d model/split rows over splits %s" % (len(summary_rows), list(EVAL_ON)))


runs discovered (19):
  instance_LSTM_seed99               INSTANCE  epochs=100 params-at-load=?
  instance_Linear_seed99             INSTANCE  epochs=100 params-at-load=?
  instance_None_seed99               INSTANCE  epochs=100 params-at-load=?
  instance_Transformer_seed99        INSTANCE  epochs=100 params-at-load=?
  instance_attention_seed99          INSTANCE  epochs=100 params-at-load=?
  pretrained_LSTM                    AUTHORS   epochs=None params-at-load=?
  pretrained_Linear                  AUTHORS   epochs=None params-at-load=?
  pretrained_None                    AUTHORS   epochs=None params-at-load=?
  pretrained_Transformer             AUTHORS   epochs=None params-at-load=?
  pretrained_attention               AUTHORS   epochs=None params-at-load=?
  stead_LSTM_seed99                  STEAD     epochs=100 params-at-load=?
  stead_Linear_seed99                STEAD     epochs=100 params-at-load=?
  stead_None_seed99                  STEAD     epochs=100 params-at-load

## Part 5: figures

Switch `RUN_FIGURES`. CPU, reads only what Part 4 wrote plus the training
histories. Every figure is saved as **PDF and PNG at 300 dpi** into `figures/`
and is backed by a CSV in `results/`, so the team can re-plot or re-read any
number.

Axes, units, legends and direct labels are set here. **Captions are not** -- the
report text is written by the team.

| Figure | Content | Feeds |
|---|---|---|
| `fig_batchdep_transformer` | median EV vs evaluation batch size, and vs row ordering | the new finding |
| `fig_loss_curves_variants` | train/validation loss per run, best-validation epoch marked | Fig. 3 of the outline |
| `fig_ev_bars_variants` | pooled median EV with 95% CI, all bottlenecks, both branches | Fig. 5 / 7, Table 3 |
| `fig_ev_vs_snr_variants` | mean clipped EV against input SNR | Fig. 8 |
| `fig_params_vs_ev` | parameter count against median EV | Table 2, discussion |
| `fig_paired_variants` | paired mean differences with CI (forest plot) | Table 5 |
| `fig_waveform_examples` | good / median / failure rows, waveform and amplitude spectrum | Fig. 4 / 9 |
| `fig_consistency_residual` | how far `earthquake + noise` departs from the input | discussion, future work |

Colours come from a palette validated for colour-vision deficiency (worst
adjacent CVD Delta E 9.1, normal-vision 22.9, both above the required floors).
Two of the four hues sit below 3:1 contrast on the chart surface, so every
series that uses them also carries a direct text label, and each figure has its
CSV.

In [9]:
if RUN_FIGURES:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    # validated categorical palette (light surface #fcfcfb)
    C_PAPER, C_NEW, C_3, C_4 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
    SERIES = [C_PAPER, C_NEW, C_3, C_4]
    INK, INK2, GRID, SURF, MUTED = "#0b0b0b", "#52514e", "#e5e4e0", "#fcfcfb", "#9a9a94"

    plt.rcParams.update({
        "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
        "font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 9,
        "axes.edgecolor": INK2, "axes.linewidth": 0.8,
        "axes.spines.top": False, "axes.spines.right": False,
        "text.color": INK, "axes.labelcolor": INK,
        "xtick.color": INK2, "ytick.color": INK2,
        "xtick.labelsize": 8, "ytick.labelsize": 8,
        "grid.color": GRID, "grid.linewidth": 0.7,
        "legend.frameon": False, "legend.fontsize": 8,
        "lines.linewidth": 1.8, "lines.markersize": 5,
        "savefig.dpi": 300, "savefig.bbox": "tight",
        "pdf.fonttype": 42, "ps.fonttype": 42,
    })

    ORDER = list(PAPER_BOTTLENECK_NAMES) + ["attention_PE", "hybrid",
                                            "attention_LSTM", "Transformer_fixed"]

    def bcolor(kind):
        return {"paper": C_PAPER, "new": C_NEW, "pretrained": C_3}.get(kind, MUTED)

    def border(ax, grid_axis="y"):
        ax.grid(True, axis=grid_axis, zorder=0)
        ax.set_axisbelow(True)

    def save_fig(fig, name):
        for ext in ("pdf", "png"):
            fig.savefig(FIGURES / ("%s.%s" % (name, ext)))
        plt.close(fig)
        print("  figure: %s (.pdf, .png)" % name)

    def load_table(name):
        p = RESULTS / name
        if not p.exists():
            print("  MISSING %s -- run the part that writes it" % name)
            return None
        t = read_csv_keep(p)
        return t if len(t) else None

    def order_key(b):
        return ORDER.index(b) if b in ORDER else len(ORDER)

    print("figure style ready; output ->", FIGURES)

figure style ready; output -> /kaggle/working/figures


In [10]:
# ============================== A. batch dependence ==============================
if RUN_FIGURES:
    bd = load_table("table_batchdep.csv")
    bp = load_table("table_batchdep_perm.csv")
    if bd is not None:
        fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.4),
                                 gridspec_kw=dict(width_ratios=[1.45, 1]))
        ax = axes[0]
        for i, b in enumerate(sorted(bd.bottleneck.unique(), key=order_key)):
            d = bd[bd.bottleneck == b].sort_values("batch_size")
            ax.plot(d.batch_size, d.median_ev_eq, "-o", color=SERIES[i % len(SERIES)],
                    label=b, zorder=3)
            ax.annotate(b, (d.batch_size.iloc[-1], d.median_ev_eq.iloc[-1]),
                        xytext=(6, 0), textcoords="offset points", va="center",
                        fontsize=8, color=SERIES[i % len(SERIES)])
        ax.set_xscale("log", base=2)
        _x0, _x1 = ax.get_xlim()
        ax.set_xlim(_x0, _x1 * 2.6)          # room for the direct labels
        ax.set_xlabel("evaluation batch size (traces)")
        ax.set_ylabel("pooled median EV, earthquake branch")
        border(ax)
        ax.legend(loc="lower right")
        ax.set_title("(a) EV against evaluation batch size", loc="left")

        ax = axes[1]
        if bp is not None:
            names = sorted(bp.bottleneck.unique(), key=order_key)
            for i, b in enumerate(names):
                d = bp[bp.bottleneck == b]
                xs = np.full(len(d), i) + np.linspace(-0.12, 0.12, len(d))
                ax.plot(xs, d.median_ev_eq, "o", color=SERIES[i % len(SERIES)],
                        markeredgecolor=SURF, markeredgewidth=0.8, zorder=3)
                ax.annotate("spread %.1e" % np.ptp(d.median_ev_eq.values),
                            (i, d.median_ev_eq.max()), xytext=(0, 9),
                            textcoords="offset points", ha="center", fontsize=7.5,
                            color=INK2)
            ax.set_xticks(range(len(names)))
            ax.set_xticklabels(names)
            ax.set_xlim(-0.5, len(names) - 0.5)
            _y0, _y1 = ax.get_ylim()
            ax.set_ylim(_y0, _y1 + 0.28 * (_y1 - _y0))   # room for the spread labels
        ax.set_ylabel("pooled median EV, earthquake branch")
        ax.set_xlabel("bottleneck")
        border(ax)
        ax.set_title("(b) %d row orderings at batch size 128" % BATCHDEP_PERMS, loc="left")
        fig.tight_layout()
        save_fig(fig, "fig_batchdep_transformer")

# ============================== B. loss curves ==============================
if RUN_FIGURES:
    _runs = discover_runs(require_epochs=0)
    hist = {}
    for mid, info in _runs.items():
        if info.get("kind") == "pretrained":
            continue          # not trained by us; no comparable loss curve
        hs = sorted(info["folder"].glob("*_Training_history.hdf5"))
        if not hs:
            continue
        with h5py.File(hs[0], "r") as f:
            hist[mid] = {k: f[k][:] for k in f.keys()}
    if hist:
        keys = sorted(hist, key=lambda m: (_runs[m]["trained_on"],
                                           order_key(_runs[m]["bottleneck"])))
        ncol = min(3, len(keys))
        nrow = int(np.ceil(len(keys) / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 2.5 * nrow),
                                 squeeze=False)
        for ax in axes.ravel()[len(keys):]:
            ax.axis("off")
        for ax, mid in zip(axes.ravel(), keys):
            h = hist[mid]
            ep = np.arange(1, len(h["loss"]) + 1)
            ax.plot(ep, h["loss"], color=C_PAPER, label="training")
            ax.plot(ep, h["val_loss"], color=C_NEW, label="validation")
            best = int(np.argmin(h["val_loss"]))
            ax.axvline(best + 1, color=MUTED, lw=0.9, ls="--", zorder=1)
            ax.annotate("best val\nepoch %d" % (best + 1), (best + 1, max(h["val_loss"])),
                        xytext=(3, -2), textcoords="offset points", fontsize=7,
                        color=INK2, va="top")
            ax.set_yscale("log")
            ax.set_title("%s  (%s)" % (_runs[mid]["bottleneck"], _runs[mid]["trained_on"]),
                         loc="left")
            ax.set_xlabel("epoch")
            ax.set_ylabel("MSE loss")
            border(ax)
        handles, labels = axes.ravel()[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=2,
                   bbox_to_anchor=(0.5, -0.02))
        fig.tight_layout()
        save_fig(fig, "fig_loss_curves_variants")
    else:
        print("  no training histories found for fig_loss_curves_variants")

# ============================== C. EV bars ==============================
if RUN_FIGURES:
    ev = load_table("table_variants_eval.csv")
    if ev is not None:
        for split in sorted(ev.test_split.unique()):
            d0 = ev[ev.test_split == split]
            fig, axes = plt.subplots(1, 2, figsize=(10.2, 0.42 * len(d0) + 1.7),
                                     sharey=True)
            # both panels share the y axis, so they must share one row order
            d = d0.sort_values("median_ev_eq")
            ypos = np.arange(len(d))
            for ax, (branch, col) in zip(axes, (("earthquake", "median_ev_eq"),
                                                ("noise", "median_ev_noise"))):
                lo = d[col] - d[col + "_lo"]
                hi = d[col + "_hi"] - d[col]
                ax.barh(ypos, d[col], height=0.62,
                        color=[bcolor(k) for k in d["kind"]],
                        xerr=[lo.values, hi.values],
                        error_kw=dict(ecolor=INK2, elinewidth=0.9, capsize=2),
                        zorder=3)
                for y, v in zip(ypos, d[col]):
                    ax.annotate("%.4f" % v, (v, y), xytext=(5, 0),
                                textcoords="offset points", va="center",
                                fontsize=7.5, color=INK)
                ax.set_yticks(ypos)
                ax.set_yticklabels(d["model_id"], fontsize=7.5)
                ax.set_xlabel("pooled median EV, %s branch" % branch)
                ax.set_xlim(min(0.0, float(d[col].min()) - 0.05),
                            max(1.0, float(d[col].max()) + 0.12))
                border(ax, grid_axis="x")
                ax.set_title("%s branch" % branch, loc="left")
            from matplotlib.patches import Patch
            _kinds = list(dict.fromkeys(d["kind"]))
            _lab = {"paper": "paper bottleneck", "new": "new bottleneck",
                    "pretrained": "authors' pretrained", "baseline": "baseline"}
            fig.legend(handles=[Patch(facecolor=bcolor(k), label=_lab.get(k, k))
                                for k in ("paper", "new", "pretrained", "baseline")
                                if k in _kinds],
                       loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.04))
            fig.suptitle("%s test split, rows ordered by earthquake-branch EV, "
                         "error bars = 95%% cluster bootstrap CI"
                         % split, x=0.01, ha="left", fontsize=9.5)
            fig.tight_layout()
            save_fig(fig, "fig_ev_bars_variants_%s" % split)

# ============================== D. EV vs input SNR ==============================
if RUN_FIGURES:
    sn = load_table("table_variants_ev_vs_snr.csv")
    if sn is not None:
        for split in sorted(sn.test_split.unique()):
            d0 = sn[sn.test_split == split]
            evt = load_table("table_variants_eval.csv")
            cand = d0.model_id.unique().tolist()
            new_ids = [m for m in cand if any(nb in m for nb in NEW_BOTTLENECK_NAMES)]
            pap_ids = [m for m in cand if m.endswith(("_attention_seed99", "_LSTM_seed99"))]
            if evt is not None:
                rank = evt[evt.test_split == split].set_index("model_id")["median_ev_eq"]
                new_ids = sorted(new_ids, key=lambda m: -rank.get(m, -9))[:2]
                pap_ids = sorted(pap_ids, key=lambda m: -rank.get(m, -9))[:2]
            pick = (pap_ids + new_ids)[:4]
            if not pick:
                continue
            fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.5), sharex=True)
            for ax, (branch, col) in zip(axes, (("earthquake", "mean_clipped_ev_eq"),
                                                ("noise", "mean_clipped_ev_noise"))):
                for i, mid in enumerate(pick):
                    d = d0[d0.model_id == mid].sort_values("snr_bin")
                    xc = np.sqrt(d.snr_lo.values * d.snr_hi.values)
                    ax.plot(xc, d[col], "-o", color=SERIES[i % len(SERIES)],
                            label=mid, zorder=3)
                    ax.annotate(mid.replace("_seed99", ""),
                                (xc[-1], d[col].values[-1]), xytext=(5, 0),
                                textcoords="offset points", va="center", fontsize=7,
                                color=SERIES[i % len(SERIES)])
                ax.set_xscale("log")
                ax.axhline(0.0, color=MUTED, lw=0.8, zorder=1)
                ax.set_xlabel("input SNR (linear amplitude factor)")
                ax.set_ylabel("mean clipped EV, %s branch" % branch)
                border(ax)
                ax.set_title("%s branch" % branch, loc="left")
            axes[0].legend(loc="lower right", fontsize=7)
            fig.suptitle("%s test split, EV clipped to [%g, 1]" % (split, EV_CLIP),
                         x=0.01, ha="left", fontsize=9.5)
            fig.tight_layout()
            save_fig(fig, "fig_ev_vs_snr_variants_%s" % split)

  figure: fig_batchdep_transformer (.pdf, .png)
  figure: fig_loss_curves_variants (.pdf, .png)
  figure: fig_ev_bars_variants_INSTANCE (.pdf, .png)
  figure: fig_ev_bars_variants_STEAD (.pdf, .png)
  figure: fig_ev_vs_snr_variants_INSTANCE (.pdf, .png)
  figure: fig_ev_vs_snr_variants_STEAD (.pdf, .png)


In [11]:
# ============================== E. parameters against EV ==============================
if RUN_FIGURES:
    ev = load_table("table_variants_eval.csv")
    if ev is not None:
        for split in sorted(ev.test_split.unique()):
            d = ev[(ev.test_split == split) & (ev.n_params > 0)]
            if not len(d):
                continue
            fig, ax = plt.subplots(figsize=(6.4, 4.0))
            for kind, lab in (("paper", "paper bottleneck"), ("new", "new bottleneck"),
                              ("pretrained", "authors' pretrained")):
                s = d[d.kind == kind]
                if not len(s):
                    continue
                ax.plot(s.n_params, s.median_ev_eq, "o", color=bcolor(kind),
                        markersize=7, markeredgecolor=SURF, markeredgewidth=0.9,
                        label=lab, zorder=3)
            for _, r in d.iterrows():
                ax.annotate(r["bottleneck"], (r["n_params"], r["median_ev_eq"]),
                            xytext=(7, -1), textcoords="offset points", fontsize=7.5,
                            color=bcolor(r["kind"]))
            ax.set_xscale("log")
            ax.set_xlabel("trainable parameters")
            ax.set_ylabel("pooled median EV, earthquake branch")
            border(ax)
            ax.legend(loc="lower right")
            ax.set_title("%s test split" % split, loc="left")
            fig.tight_layout()
            save_fig(fig, "fig_params_vs_ev_%s" % split)

# ============================== F. paired differences ==============================
if RUN_FIGURES:
    pr = load_table("table_variants_paired.csv")
    if pr is not None:
        for split in sorted(pr.test_split.unique()):
            d0 = pr[pr.test_split == split]
            branches = sorted(d0.branch.unique())
            _rows = int(d0.groupby('branch').size().max())   # rows per panel, not total
            fig, axes = plt.subplots(1, len(branches),
                                     figsize=(5.4 * len(branches), 0.40 * _rows + 1.8),
                                     squeeze=False)
            for ax, branch in zip(axes[0], branches):
                d = d0[d0.branch == branch].copy()
                d["label"] = [("%s - %s  [%s]"
                               % (a.replace("_seed99", "").split("_", 1)[1],
                                  b.replace("_seed99", "").split("_", 1)[1],
                                  t.lower()))
                              + ("  *" if s else "")
                              for a, b, t, s in zip(d.model_a, d.model_b,
                                                    d.trained_on, d.significant_at_95)]
                d = d.sort_values("mean_diff")
                y = np.arange(len(d))
                ax.errorbar(d.mean_diff, y,
                            xerr=[(d.mean_diff - d.ci_lo).values,
                                  (d.ci_hi - d.mean_diff).values],
                            fmt="o", color=C_PAPER, ecolor=INK2, elinewidth=1.0,
                            capsize=2, markersize=5, markeredgecolor=SURF, zorder=3)
                ax.axvline(0.0, color=MUTED, lw=1.0, zorder=1)
                ax.set_yticks(y)
                ax.set_yticklabels(d["label"], fontsize=7.5)
                ax.set_xlabel("difference in mean clipped EV (A - B)")
                border(ax, grid_axis="x")
                ax.set_title("%s branch" % branch, loc="left")
            fig.suptitle("%s test split; bars = 95%% cluster bootstrap CI, "
                         "* = CI excludes zero" % split,
                         x=0.01, ha="left", fontsize=9.5)
            fig.tight_layout()
            save_fig(fig, "fig_paired_variants_%s" % split)

# ============================== G. example waveforms ==============================
if RUN_FIGURES:
    for split in EVAL_ON:
        p = RESULTS / ("examples_%s.npz" % split)
        if not p.exists():
            print("  MISSING examples_%s.npz" % split)
            continue
        z = np.load(p, allow_pickle=False)
        labels = [str(s) for s in z["labels"]]
        shown = [str(s) for s in z["shown"]]
        comp, fs = 2, 10.0                       # vertical component as stored, 10 Hz
        PRED_LS = ["--", "-."]                   # identity is never carried by colour alone
        t = np.arange(z["x"].shape[1]) / fs
        fig, axes = plt.subplots(len(labels), 2, figsize=(10.0, 2.3 * len(labels)),
                                 gridspec_kw=dict(width_ratios=[1.75, 1]))
        for i, lab in enumerate(labels):
            axw, axs = axes[i, 0], axes[i, 1]
            axw.plot(t, z["x"][i, :, comp], color=MUTED, lw=0.9,
                     label="input (mixture)", zorder=2)
            axw.plot(t, z["y"][i, :, comp], color=SERIES[0], lw=1.5,
                     label="true earthquake", zorder=3)
            for j, mid in enumerate(shown):
                k = "pred_eq__" + mid
                if k in z:
                    axw.plot(t, z[k][i, :, comp], color=SERIES[1 + j], lw=1.3,
                             ls=PRED_LS[j % len(PRED_LS)],
                             label="predicted, " + mid.replace("_seed99", ""), zorder=4)
            axw.set_ylabel("amplitude\n(normalised)")
            axw.set_title("%s   EV(ref) = %.3f" % (lab, float(z["ev_ref"][i])), loc="left")
            border(axw)
            if i == len(labels) - 1:
                axw.set_xlabel("time (s)")

            def spec(v):
                A = np.abs(np.fft.rfft(v))
                return np.fft.rfftfreq(len(v), 1.0 / fs), A
            f0, A = spec(z["x"][i, :, comp])
            axs.plot(f0, A, color=MUTED, lw=0.9, zorder=2)
            f0, A = spec(z["y"][i, :, comp])
            axs.plot(f0, A, color=SERIES[0], lw=1.5, zorder=3)
            for j, mid in enumerate(shown):
                k = "pred_eq__" + mid
                if k in z:
                    f0, A = spec(z[k][i, :, comp])
                    axs.plot(f0, A, color=SERIES[1 + j], lw=1.3,
                             ls=PRED_LS[j % len(PRED_LS)], zorder=4)
            axs.set_yscale("log")
            axs.set_xlim(0, fs / 2)
            axs.set_ylabel("amplitude spectrum")
            border(axs)
            if i == len(labels) - 1:
                axs.set_xlabel("frequency (Hz)")
        h, l = axes[0, 0].get_legend_handles_labels()
        fig.legend(h, l, loc="lower center", ncol=min(4, len(l)),
                   bbox_to_anchor=(0.5, -0.02))
        fig.suptitle("%s test split, vertical component, reference model %s"
                     % (split, str(z["reference"])), x=0.01, ha="left", fontsize=9.5)
        fig.tight_layout()
        save_fig(fig, "fig_waveform_examples_%s" % split)

# ============================== H. consistency residual ==============================
if RUN_FIGURES:
    ev = load_table("table_variants_eval.csv")
    if ev is not None:
        for split in sorted(ev.test_split.unique()):
            d = ev[(ev.test_split == split) & (ev.n_params > 0)].copy()
            d = d.sort_values("consistency_residual")
            if not len(d):
                continue
            fig, ax = plt.subplots(figsize=(7.2, 0.34 * len(d) + 1.6))
            y = np.arange(len(d))
            ax.barh(y, d.consistency_residual, height=0.62,
                    color=[bcolor(k) for k in d["kind"]], zorder=3)
            for yy, v in zip(y, d.consistency_residual):
                ax.annotate("%.2e" % v, (v, yy), xytext=(5, 0),
                            textcoords="offset points", va="center",
                            fontsize=7.5, color=INK)
            ax.set_yticks(y)
            ax.set_yticklabels(d.model_id, fontsize=7.5)
            ax.set_xscale("log")
            ax.set_xlabel(r"median  $\langle(\hat{y}_{eq}+\hat{y}_{noise}-x)^2\rangle\ /\ "
                          r"\langle x^2\rangle$")
            border(ax, grid_axis="x")
            from matplotlib.patches import Patch
            _kinds = list(dict.fromkeys(d["kind"]))
            _lab = {"paper": "paper bottleneck", "new": "new bottleneck",
                    "pretrained": "authors' pretrained"}
            ax.legend(handles=[Patch(facecolor=bcolor(k), label=_lab[k])
                               for k in ("paper", "new", "pretrained") if k in _kinds],
                      loc="lower right")
            ax.set_title("%s test split" % split, loc="left")
            fig.tight_layout()
            save_fig(fig, "fig_consistency_residual_%s" % split)

if RUN_FIGURES:
    print("\nfigures written to", FIGURES)
    for p in sorted(FIGURES.glob("*.pdf")):
        print("   ", p.name)
    log("Part 5: %d figures" % len(list(FIGURES.glob("*.pdf"))))

  figure: fig_params_vs_ev_INSTANCE (.pdf, .png)
  figure: fig_params_vs_ev_STEAD (.pdf, .png)
  figure: fig_paired_variants_INSTANCE (.pdf, .png)
  figure: fig_paired_variants_STEAD (.pdf, .png)
  figure: fig_waveform_examples_STEAD (.pdf, .png)
  figure: fig_waveform_examples_INSTANCE (.pdf, .png)
  figure: fig_consistency_residual_INSTANCE (.pdf, .png)
  figure: fig_consistency_residual_STEAD (.pdf, .png)

figures written to /kaggle/working/figures
    fig_batchdep_transformer.pdf
    fig_consistency_residual_INSTANCE.pdf
    fig_consistency_residual_STEAD.pdf
    fig_ev_bars_variants_INSTANCE.pdf
    fig_ev_bars_variants_STEAD.pdf
    fig_ev_vs_snr_variants_INSTANCE.pdf
    fig_ev_vs_snr_variants_STEAD.pdf
    fig_loss_curves_variants.pdf
    fig_paired_variants_INSTANCE.pdf
    fig_paired_variants_STEAD.pdf
    fig_params_vs_ev_INSTANCE.pdf
    fig_params_vs_ev_STEAD.pdf
    fig_waveform_examples_INSTANCE.pdf
    fig_waveform_examples_STEAD.pdf
- 2026-11-17 17:45:20  Part 5: 14 fi

## After each session

Save a version, then download `results/`, `figures/` and `logs/`.

| Session | Switches | Send me |
|---|---|---|
| Check before training | `RUN_SELFTEST` | the printed output of Part 1 |
| The batch-size finding | `RUN_SELFTEST`, `RUN_BATCHDEP` | `table_batchdep*.csv` |
| Training (GPU) | `RUN_SMOKE` once, then `RUN_TRAIN` | the run folder, and `run_info.json` |
| Evaluation (CPU) | `RUN_EVAL` | `table_variants_eval.csv`, `table_variants_paired.csv`, `table_variants_ev_vs_snr.csv` |
| Figures (CPU) | `RUN_FIGURES` | the PNGs, so I can check the axes and numbers against the tables |

Do not send `.hdf5`, `.pth` or `per_sample_*.csv.gz` files unless a specific
check needs one.

## Reminders

- One variant per `RUN_TRAIN` session. Upload each finished run folder to the
  runs dataset before the evaluation session, or the evaluation will not find it.
- One training seed (99). With 28 earthquakes in the INSTANCE test split, a
  difference below about 0.02 EV is not distinguishable from noise; the paired
  table says which differences have a CI that excludes zero. The CIs cover
  test-sample variation only, not training randomness.
- `EVAL_BATCH` is recorded in `table_variants_eval.csv`. Keep it at 128 for
  every run, and quote it next to any Transformer number.
- Keep `table_E1_E2_E3.csv` from Phase 2 as it is. The new variants have no E1
  or E2 column unless they are trained on STEAD, so they belong in their own
  table.
- The deviation log needs new lines for this phase: the three `step3` edits, the
  new bottleneck modules, the corrected softmax axis in the reimplemented
  `attention_LSTM` / `hybrid`, and `batch_first` / `dim_feedforward` in
  `Transformer_fixed`.